# Dipole deconvolution + denoising — claude-final iter3

**Unrolled HQS (Half-Quadratic Splitting) + plug-and-play denoiser prior.**

iter1(26.514) / iter5(26.538) 의 단일 pass residual 구조를 **반복 구조**로 교체한다.

## 왜 구조를 바꾸는가 — 실측 근거

`colab_day3.ipynb` (별도 트랙, unrolled DRUNet + σ-map) 가 **29.25 dB** 를 냈다.
claude-final 과 noise type 별로 비교하면 격차의 출처가 한 곳에 몰려 있다.

| | ALL | gaussian | rician | uniform | **S&P** |
|---|---|---|---|---|---|
| day3 unrolled | **29.25** | 30.03 | 22.47 | 29.23 | **35.26** |
| iter5 C+TTA (단일 pass) | 26.538 | 28.497 | 21.329 | 27.202 | 29.123 |
| **Δ** | **+2.71** | +1.53 | +1.14 | +2.03 | **+6.14** |

S&P 의 +6.14 dB 가 ALL 에 기여하는 몫은 `6.14/4 = 1.54 dB` 로 **전체 격차의 57%** 다.

`iter5_review.md` 3.3 에서 iter5 가 `median_wiener` 채널을 추가해 S&P 를 **+0.304** 올린 것을 확인했다.
같은 문제를 day3 는 구조로 풀어 **+6.14** 를 얻었다. **채널 추가로는 도달 불가능한 격차다.**

impulse 는 반복마다 조금씩 제거되고, 닫힌 해 DC step 이 매번 물리를 다시 강제한다.
단일 pass 는 이 반복이 없다.

## Unrolled HQS

`y = A x + n`, `A` 는 k-space 대각 연산자 `D(k)` 이므로 data-consistency 부분문제가 **닫힌 해**를 갖는다.

$$\min_x \|Ax - y\|^2 + \lambda\|x - z\|^2
\quad\Longrightarrow\quad
X(k) = \frac{D(k)\,Y(k) + \lambda Z(k)}{D(k)^2 + \lambda}$$

```
x₀ = Wiener(y, K)                          ← 해석적 초기값
반복 t = 1..4:
    z  = denoiser(x, y, σ̂, 부가정보)        ← prior step (학습)
    X  = (D·Y + λ_t·Z) / (D² + λ_t)        ← DC step (닫힌 해, 학습 파라미터 λ_t)
    x  = ifft(X).real
출력 = x
```

### 하한 보장

`λ → ∞` 이면 `X → Z` 다. denoiser 의 마지막 conv 를 **zero-init** 하면 `z = x` 이므로
초기 상태의 반복은 항등에 가깝다. 배율 오차는 정확히 계산된다.

$$\text{ratio} = \frac{X}{x_0} = 1 + \frac{K}{D^2 + \lambda}$$

| `λ_init` | 1 iteration 배율 | 4 iteration 누적 |
|---|---|---|
| 0.05 | 1.036 ~ 1.356 | +237.8% |
| 1.0 | 1.012 ~ 1.018 | +7.3% |
| 4.0 | 1.004 ~ 1.004 | +1.8% |
| **10.0** | **1.0017 ~ 1.0018** | **+0.71%** |

`λ_init = 10` 을 쓴다. 학습 시작 시점 출력이 `Wiener(y, K)` 와 0.71% 이내로 일치하므로
**conventional 성능이 하한으로 보장**된다. `λ_t` 는 학습되며 필요한 만큼 작아진다.

## day3 에 없는 것을 얹는다

29.25 를 넘으려면 day3 가 하지 않은 것이 필요하다.

| # | 추가 | 근거 |
|---|---|---|
| **1** | **Rician 판별 신호** (`음수 픽셀 비율` 상수맵) | rician 측정값은 `\|Ax+n\|` 이라 **엄격히 비음수**. 실측: rician 25/25장 `min ≥ 0.0002`, gaussian 25/25 · uniform 21/25 · S&P 18/25 는 음수 포함. `min(y) ≥ 0` 만으로 rician 을 거의 완벽히 구분한다. day3 는 rician 이 22.47 로 최하위 |
| **2** | **impulse 위치 맵** (`\|y − median₃(y)\| / σ̂`) | S&P 를 **공간적으로 국소화**해 준다. day3 는 이 신호가 없다 |
| **3** | **SSIM + Sobel 손실** | `iter5_review.md` 3.3 — `ssim_weight` 0.5 + Sobel 0.1 이 **4종 전부** SSIM 을 올렸다(+0.0053~+0.0109). day3 는 `charbonnier` 단독 |
| **4** | **deep supervision** | 중간 반복 출력도 감독해 4단 unrolling 의 수렴을 돕는다 |
| **5** | **출력 clamp [0,1]** | day3 입력 PSNR(gaussian 7.91)이 내 측정(8.156)보다 낮은 것은 clamp 미적용으로 보인다. label 이 [0,1] 이므로 무료 이득 |
| **6** | **flip TTA** | `A(flip x) = flip(Ax)` 오차 1.2e-07 실측. iter1 에서 +0.224 dB |

## σ ablation 을 제대로 한다

day3 의 σ ablation 에서 **"σ 뒤섞음" 이 정상값과 소수점 4자리까지 동일(29.25 / 0.8777)** 했다.
per-image σ 를 실제로 쓴다면 뒤섞으면 성능이 떨어져야 한다.
`σ=0`(15.64) 과 `σ×2`(22.45) 는 결과를 바꾸므로, 뒤섞기가 모델에 도달하지 않는 경로로 추정된다.

iter3 는 σ̂ 를 **입력 텐서에서 직접 치환**해 이 ablation 을 정확히 수행한다.

## 0. 환경 준비

In [ ]:
# Colab 이면 Drive mount, 로컬이면 건너뛴다
try:
    from google.colab import drive

    drive.mount("/content/drive")
    IN_COLAB = True
except Exception as err:
    print(f"Colab 환경이 아니다 ({type(err).__name__}). Drive mount 를 건너뛴다.")
    IN_COLAB = False

In [ ]:
import glob
import json
import math
import os
import random
import shutil
import tempfile
import time
import warnings
import zlib
from collections.abc import Callable
from dataclasses import asdict, dataclass, field
from datetime import datetime
from enum import Enum, IntEnum
from functools import lru_cache
from pathlib import Path
from typing import Any, Literal

import matplotlib.pyplot as plt
import numpy as np
import torch
from torch import Tensor, nn
from torch.nn import functional
from torch.optim import Adam, AdamW
from torch.utils.data import DataLoader, Dataset
from tqdm.auto import tqdm

warnings.filterwarnings("ignore")

# ---- ROOT 자동 탐색: dataset/train 을 가진 디렉토리를 찾는다 ----
CANDIDATES = [
    Path("/content/drive/MyDrive/DS2026/20260831-pjt5-이종호/실습"),
    Path("/content/drive/MyDrive/실습프로젝트"),
    Path("/content/drive/MyDrive"),
    Path("C:/hong/project-5/ref"),
    Path.cwd() / "ref",
    Path.cwd(),
    Path.cwd().parent / "ref",
]

ROOT = None
for _c in CANDIDATES:
    if (_c / "dataset" / "train").is_dir():
        ROOT = _c
        break
if ROOT is None:
    for _c in CANDIDATES:
        if not _c.exists():
            continue
        for _hit in _c.glob("**/dataset/train"):
            ROOT = _hit.parent.parent
            break
        if ROOT is not None:
            break
if ROOT is None:
    raise FileNotFoundError(f"dataset/train 을 찾지 못했다. CANDIDATES 를 수정할 것: {CANDIDATES}")

DATA_SRC = ROOT / "dataset"
WORK_ROOT = ROOT / "claude-final"
WORK_ROOT.mkdir(parents=True, exist_ok=True)

# packed cache 는 항상 로컬 SSD 에 둔다 (Colab: /content)
CACHE_DIR = Path("/content/cache") if IN_COLAB else Path(tempfile.gettempdir()) / "pjt5_cache"
CACHE_DIR.mkdir(parents=True, exist_ok=True)

print("ROOT     :", ROOT)
print("DATA_SRC :", DATA_SRC)
print("WORK_ROOT:", WORK_ROOT, "(ckpt/log 저장)")
print("CACHE_DIR:", CACHE_DIR, "(학습 중 읽는 곳)")
print("GPU      :", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "없음 (CPU)")

In [ ]:
# ---- 상수: 조교 제공 코드와 동일하게 유지한다 ----
VOXEL_SIZE: tuple[float, float] = (1.0, 1.0)
B0_DIR: tuple[float, float] = (0.0, 1.0)  # 실측 확인: test 전체가 (0, 1)

NOISE_RANGES: dict[str, tuple[float, float]] = {
    "gaussian": (0.0, 0.1),
    "rician": (0.0, 0.15),
    "uniform": (0.0, 0.2),
    "salt_and_pepper": (0.0, 0.2),
}
NOISE_NAMES: list[str] = list(NOISE_RANGES.keys())

# 규칙: clean 1장당 orientation ≤ 6, orientation 당 noise ≤ 2 → corrupted ≤ 12
RULE_MAX_VARIANTS: int = 12


@dataclass
class Config:
    # split
    train_split: str = "train"
    valid_split: str = "val"
    test_measure_split: str = "test_deconv_noise"
    test_label_split: str = "test_label"
    test_blur_split: str = "test_deconv_only"  # forward model 자기검증용 (없어도 됨)

    # 데이터 생성 규칙
    variant_mode: Literal["pool", "onthefly"] = "pool"
    variant_pool: int = RULE_MAX_VARIANTS
    augment_flip: bool = True  # False 로 두면 clean 당 corrupted 가 정확히 variant_pool 개가 된다

    # validation (층화)
    valid_l1_only: bool = True
    val_sigma_levels: int = 3
    valid_batch: int = 16

    # 학습 공통
    num_workers: int = 2
    amp: bool = True
    clip_output: bool = True
    seed: int = 1234

    # 빠른 스모크 테스트
    max_train_images: int | None = None

    device: torch.device = field(default_factory=lambda: torch.device("cuda" if torch.cuda.is_available() else "cpu"))


config = Config()
if os.name == "nt":
    config.num_workers = 0  # Windows 로컬에서는 worker 가 자주 멈춘다

torch.manual_seed(config.seed)
np.random.seed(config.seed)
random.seed(config.seed)
torch.backends.cudnn.benchmark = True

print("device:", config.device)
for _k, _v in asdict(config).items():
    print(f"  {_k}: {_v}")

## 1. Drive → `/content` SSD packed 캐시

Drive 에서 `.npy` 를 7,468번 개별로 여는 것이 학습 속도의 가장 큰 병목이다.
한 번만 순차로 읽어 **하나의 `(N, 256, 256) float32 memmap** 으로 합쳐두면
이후 모든 epoch 이 로컬 SSD 의 memmap 슬라이싱으로 끝난다.

- `train` 7,268장 → 약 1.9 GB
- 이미 캐시가 있으면 건너뛴다 (`force=True` 로 재생성)

In [ ]:
def build_cache(split: str, out_name: str | None = None, force: bool = False) -> tuple[Path, Path] | None:
    # DATA_SRC/split/*.npy 를 하나의 packed .npy + names.json 으로 만든다
    out_name = out_name or split
    npy_path = CACHE_DIR / f"{out_name}.npy"
    names_path = CACHE_DIR / f"{out_name}_names.json"

    src = DATA_SRC / split
    if not src.is_dir():
        print(f"[skip] {split}: 디렉토리 없음")
        return None

    files = sorted(src.glob("*.npy"))
    if not files:
        # 한 단계 더 깊게 배포된 경우 (예: test_noise_only/test_noise_only/*.npy)
        nested = src / split
        files = sorted(nested.glob("*.npy")) if nested.is_dir() else []
    if not files:
        print(f"[skip] {split}: npy 없음")
        return None

    if npy_path.exists() and names_path.exists() and not force:
        names = json.loads(names_path.read_text(encoding="utf-8"))
        if len(names) == len(files):
            print(f"[cached] {out_name}: {len(names)}장")
            return npy_path, names_path

    probe = np.load(files[0]).squeeze()
    h, w = probe.shape
    arr = np.lib.format.open_memmap(npy_path, mode="w+", dtype=np.float32, shape=(len(files), h, w))
    for i, f in enumerate(tqdm(files, desc=f"cache {out_name}", unit="img")):
        arr[i] = np.load(f).squeeze().astype(np.float32)
    arr.flush()
    del arr

    names_path.write_text(json.dumps([f.name for f in files], ensure_ascii=False), encoding="utf-8")
    size_gb = npy_path.stat().st_size / 1024**3
    print(f"[built] {out_name}: {len(files)}장 {h}x{w} → {npy_path} ({size_gb:.2f} GB)")
    return npy_path, names_path


t0 = time.time()
build_cache(config.train_split)
build_cache(config.valid_split)
build_cache(config.test_measure_split, "test_measure")
build_cache(config.test_label_split, "test_label")
build_cache(config.test_blur_split, "test_blur")  # 없으면 skip

# noise_meta.json 은 결과 분석에만 쓴다 (학습에는 사용 금지)
for _meta_split, _meta_name in [(config.test_measure_split, "noise_meta.json")]:
    _src_meta = DATA_SRC / _meta_split / _meta_name
    if _src_meta.exists():
        shutil.copy2(_src_meta, CACHE_DIR / _meta_name)
        print(f"[copy] {_meta_name}")
print(f"캐시 준비 완료: {time.time() - t0:.1f}s")

In [ ]:
class PackedSplit:
    # packed .npy memmap + 파일명 리스트. worker 마다 lazy 하게 열어 fork 안전성을 확보한다
    def __init__(self, name: str, l1_only: bool = False, limit: int | None = None) -> None:
        self.npy_path = CACHE_DIR / f"{name}.npy"
        names = json.loads((CACHE_DIR / f"{name}_names.json").read_text(encoding="utf-8"))
        idx = list(range(len(names)))
        if l1_only:
            idx = [i for i in idx if names[i].startswith("L1_")]
        if limit is not None:
            idx = idx[:limit]
        self.index = idx
        self.names = [names[i] for i in idx]
        self._mm: np.ndarray | None = None

    def __len__(self) -> int:
        return len(self.index)

    def get(self, i: int) -> np.ndarray:
        if self._mm is None:
            self._mm = np.load(self.npy_path, mmap_mode="r")
        return np.asarray(self._mm[self.index[i]])

    def name_to_pos(self) -> dict[str, int]:
        return {n: i for i, n in enumerate(self.names)}


_p = PackedSplit(config.train_split)
print(f"train {len(_p)}장 | L1 {sum(n.startswith('L1_') for n in _p.names)} / L8 {sum(n.startswith('L8_') for n in _p.names)}")
_v = PackedSplit(config.valid_split)
print(f"val   {len(_v)}장 | L1 {sum(n.startswith('L1_') for n in _v.names)} / L8 {sum(n.startswith('L8_') for n in _v.names)}")
_tm, _tl = PackedSplit("test_measure"), PackedSplit("test_label")
print(f"test  measure {len(_tm)}장 / label {len(_tl)}장 | 파일명 일치: {_tm.names == _tl.names}")
_s = _p.get(0)
print(f"sample: {_s.shape} {_s.dtype} range [{_s.min():.3f}, {_s.max():.3f}]")
del _p, _v, _tm, _tl

## 2. Forward model : dipole convolution → noise

$$y = \mathcal{N}\big(\underbrace{\mathcal{F}^{-1}\{D(k)\,\mathcal{F}\{x\}\}}_{\text{dipole convolution}}\big),\qquad
D(k) = \frac{1}{3} - \frac{(k\cdot\hat{B_0})^2}{|k|^2}$$

조교 제공 코드와 **완전히 동일한 구현**을 쓴다. 아래 셀에서 제공 데이터와 대조해 검증한다.

`B0_dir=(0,1)` 이므로 magic angle(약 54.7도) 원뿔에서 `D≈0` 이고 그 주파수 성분은 원리적으로 복원 불가능하다.
단 분모의 `+1e-8` 때문에 정확히 0 은 아니다.

In [ ]:
@lru_cache(maxsize=16)
def dipole_kernel(
    matrix_size: tuple[int, int],
    voxel_size: tuple[float, float] = VOXEL_SIZE,
    B0_dir: tuple[float, float] = B0_DIR,
) -> torch.Tensor:
    y = np.arange(-matrix_size[1] / 2, matrix_size[1] / 2, 1)
    x = np.arange(-matrix_size[0] / 2, matrix_size[0] / 2, 1)
    Y, X = np.meshgrid(y, x)

    X = X / (matrix_size[0] * voxel_size[0])
    Y = Y / (matrix_size[1] * voxel_size[1])

    D = 1 / 3 - (X * B0_dir[0] + Y * B0_dir[1]) ** 2 / (X**2 + Y**2 + 1e-8)
    D = np.fft.fftshift(D)
    return torch.tensor(D, dtype=torch.float32)


def dipole_forward(img: Tensor) -> Tensor:
    kernel = dipole_kernel(tuple(img.shape[-2:])).to(img.device)
    img_k = torch.fft.fftn(img.float(), dim=(-2, -1))
    return torch.fft.ifftn(img_k * kernel, dim=(-2, -1)).real


dipole_adjoint = dipole_forward  # D 가 real·even 이므로 A^T = A

_D = dipole_kernel((256, 256))
print(f"D range [{_D.min():.4f}, {_D.max():.4f}]")
print(f"|D| < 0.01  비율 {float((_D.abs() < 0.01).float().mean()) * 100:.2f}%")
print(f"|D| 최솟값  {float(_D.abs().min()):.3e}  (0 이 아니므로 원리적으로는 가역)")

In [ ]:
class NoisyType(str, Enum):
    Gaussian = "gaussian"
    Rician = "rician"
    Uniform = "uniform"
    SaltAndPepper = "salt_and_pepper"

    @classmethod
    def from_string(cls, value: str) -> "NoisyType":
        try:
            return cls(value)
        except ValueError as err:
            raise ValueError(f"Invalid NoisyType value: {value}. Must be one of {list(cls)} : {err}") from err


Gen = torch.Generator | None


def gaussian_noise(img: Tensor, sigma: float, generator: Gen = None) -> Tensor:
    noise = torch.empty_like(img).normal_(0.0, 1.0, generator=generator) * sigma
    return img + noise


def rician_noise(img: Tensor, sigma: float, generator: Gen = None) -> Tensor:
    noise_real = torch.empty_like(img).normal_(0.0, 1.0, generator=generator) * sigma
    noise_imag = torch.empty_like(img).normal_(0.0, 1.0, generator=generator) * sigma
    return torch.abs(img + noise_real + 1j * noise_imag)


def uniform_noise(img: Tensor, sigma: float, generator: Gen = None) -> Tensor:
    return img + (torch.empty_like(img).uniform_(0.0, 1.0, generator=generator) * 2.0 - 1.0) * sigma


def salt_and_pepper_noise(img: Tensor, sigma: float, generator: Gen = None) -> Tensor:
    salt_prob = sigma / 2
    pepper_prob = sigma / 2
    noisy_img = img.clone()
    total_pixels = img.numel()

    num_salt = int(total_pixels * salt_prob)
    coords = [torch.randint(0, dim, (num_salt,), generator=generator) for dim in img.shape]
    noisy_img[tuple(coords)] = img.max()

    num_pepper = int(total_pixels * pepper_prob)
    coords = [torch.randint(0, dim, (num_pepper,), generator=generator) for dim in img.shape]
    noisy_img[tuple(coords)] = 0

    return noisy_img


NOISE_FUNC: dict[NoisyType, Callable[..., Tensor]] = {
    NoisyType.Gaussian: gaussian_noise,
    NoisyType.Rician: rician_noise,
    NoisyType.Uniform: uniform_noise,
    NoisyType.SaltAndPepper: salt_and_pepper_noise,
}


class NoiseSimulator:
    # noise 종류 하나 + sigma 하나를 고정해서 적용한다
    def __init__(self, noisy_type: NoisyType, sigma: float) -> None:
        self.noisy_type = noisy_type
        self.sigma = sigma

    def __call__(self, img: Tensor, generator: Gen = None) -> Tensor:
        return NOISE_FUNC[self.noisy_type](img, self.sigma, generator)


def apply_noise(img: Tensor, noise_type: str, sigma: float, seed: int | None = None) -> Tensor:
    gen = None if seed is None else torch.Generator().manual_seed(int(seed) % (2**63 - 1))
    return NoiseSimulator(NoisyType.from_string(noise_type), sigma)(img, gen)


def degrade(label2d: Tensor, noise_type: str, sigma: float, seed: int | None = None) -> tuple[Tensor, Tensor]:
    # label(2D) -> (blur = Ax, measure = N(Ax)).  실측 확인된 순서: dipole conv -> noise
    blur = dipole_forward(label2d)
    measure = apply_noise(blur, noise_type, sigma, seed)
    return blur, measure


print("noise 함수 4종 등록 완료:", [n.value for n in NoisyType])

### 2-1. Forward model 자기검증

내 `degrade()` 가 조교가 배포한 데이터를 재현하는지 대조한다. 이게 틀리면 학습 전체가 무의미하다.

1. `dipole_forward(test_label)` vs `test_deconv_only` → 일치해야 한다
2. `test_deconv_noise − test_deconv_only` 잔차 통계 vs `noise_meta.json` 의 선언값
   - gaussian: `std ≈ σ`
   - uniform: `std ≈ σ/√3`
   - rician: 양의 bias
   - salt_and_pepper: sparse impulse

In [ ]:
_tl = PackedSplit("test_label")
_tm = PackedSplit("test_measure")
_tb_path = CACHE_DIR / "test_blur.npy"

NOISE_META_PATH = CACHE_DIR / "noise_meta.json"
TEST_NOISE_META: dict[str, dict] = {}
if NOISE_META_PATH.exists():
    TEST_NOISE_META = {m["file"]: m for m in json.loads(NOISE_META_PATH.read_text(encoding="utf-8"))}
    from collections import Counter

    print("test noise 분포:", dict(Counter(m["noise_type"] for m in TEST_NOISE_META.values())))

if _tb_path.exists():
    _tb = PackedSplit("test_blur")
    _pos = _tb.name_to_pos()
    print()
    print("[1] dipole_forward(label) vs test_deconv_only")
    for _n in _tl.names[:5]:
        lab = torch.from_numpy(_tl.get(_tl.name_to_pos()[_n]))
        only = torch.from_numpy(_tb.get(_pos[_n]))
        sim = dipole_forward(lab)
        err = (sim - only).abs()
        print(f"  {_n[:24]} max_err={err.max():.3e}  rmse={err.pow(2).mean().sqrt():.3e}")

    print()
    print("[2] measure - blur 잔차 통계 vs noise_meta")
    print(f"  {'file':<26}{'type':<18}{'sigma':>8}{'resid_std':>11}{'resid_mean':>12}{'expected_std':>14}")
    for _n in _tl.names[:8]:
        only = torch.from_numpy(_tb.get(_pos[_n]))
        noisy = torch.from_numpy(_tm.get(_tm.name_to_pos()[_n]))
        r = noisy - only
        meta = TEST_NOISE_META.get(_n, {})
        nt, sg = meta.get("noise_type", "?"), meta.get("sigma", float("nan"))
        exp = {"gaussian": sg, "uniform": sg / math.sqrt(3), "rician": sg, "salt_and_pepper": float("nan")}.get(nt, float("nan"))
        print(f"  {_n[:24]:<26}{nt:<18}{sg:>8.4f}{r.std():>11.4f}{r.mean():>+12.4f}{exp:>14.4f}")
    del _tb
else:
    print("test_deconv_only 가 없어 [1],[2] 검증을 건너뛴다 (학습에는 영향 없음)")

print()
print("[3] 내 apply_noise 재현성 확인 (같은 seed → 같은 결과)")
_x = torch.from_numpy(_tl.get(0))
for _nt in NOISE_NAMES:
    a = apply_noise(_x, _nt, 0.1, seed=42)
    b = apply_noise(_x, _nt, 0.1, seed=42)
    c = apply_noise(_x, _nt, 0.1, seed=43)
    print(f"  {_nt:<18} seed 동일→max diff {float((a - b).abs().max()):.2e} | seed 다름→mean|a-c| {float((a - c).abs().mean()):.4f}")
del _tl, _tm

## 3. Metric (PSNR / SSIM)

조교 제공 구현을 **그대로** 쓴다. 채점 코드와 동일해야 하므로 수정하지 않는다.
`calculate_psnr` 의 peak 는 `ref` 의 최댓값이다 (label 이 항상 1.0 은 아니다).

In [ ]:
IMG_DIM: int = 4


class SSIMcal(torch.nn.Module):
    def __init__(self, win_size: int = 11, k1: float = 0.01, k2: float = 0.03):
        super().__init__()
        self.win_size = win_size
        self.k1, self.k2 = k1, k2
        self.register_buffer("w", torch.ones(1, 1, win_size, win_size) / win_size**2)
        np_ = win_size**2
        self.cov_norm = np_ / (np_ - 1)

    def forward(self, img: Tensor, ref: Tensor, data_range: Tensor) -> Tensor:
        data_range = data_range[:, None, None, None]
        C1 = (self.k1 * data_range) ** 2
        C2 = (self.k2 * data_range) ** 2

        w = self.w.to(img.device)
        ux = functional.conv2d(img, w)
        uy = functional.conv2d(ref, w)
        uxx = functional.conv2d(img * img, w)
        uyy = functional.conv2d(ref * ref, w)
        uxy = functional.conv2d(img * ref, w)

        vx = self.cov_norm * (uxx - ux * ux)
        vy = self.cov_norm * (uyy - uy * uy)
        vxy = self.cov_norm * (uxy - ux * uy)

        A1 = 2 * ux * uy + C1
        A2 = 2 * vxy + C2
        B1 = ux**2 + uy**2 + C1
        B2 = vx + vy + C2

        return torch.mean((A1 * A2) / (B1 * B2), dim=[2, 3], keepdim=True)


ssim_cal = SSIMcal()


def calculate_ssim(img: Tensor, ref: Tensor, mask: Tensor | None = None) -> Tensor:
    if not (img.dim() == IMG_DIM and ref.dim() == IMG_DIM):
        raise ValueError("All tensors must be 4D.")

    if mask is None:
        img_mask, ref_mask = img, ref
    else:
        if mask.dim() != IMG_DIM:
            raise ValueError("Mask must be 4D.")
        img_mask, ref_mask = img * mask, ref * mask

    ones = torch.ones(ref.shape[0], device=ref.device)
    return ssim_cal.forward(img_mask, ref_mask, ones)


def calculate_psnr(img: Tensor, ref: Tensor, mask: Tensor | None = None) -> Tensor:
    if not (img.dim() == IMG_DIM and ref.dim() == IMG_DIM):
        raise ValueError("All tensors must be 4D.")

    if mask is not None:
        if mask.dim() != IMG_DIM:
            raise ValueError("Mask must be 4D.")
        img_mask, ref_mask = img * mask, ref * mask
        mse = torch.sum((img_mask - ref_mask) ** 2, dim=(1, 2, 3)) / torch.sum(mask, dim=(1, 2, 3))
    else:
        mse = torch.mean(functional.mse_loss(img, ref, reduction="none"), dim=(1, 2, 3), keepdim=True)

    img_max = torch.amax(ref, dim=(1, 2, 3), keepdim=True)
    return 10 * torch.log10(img_max**2 / (mse + 1e-12))


def finalize(pred: Tensor) -> Tensor:
    # 모든 방법에 동일하게 적용하는 후처리. label 이 [0,1] 이므로 clip 한다
    return pred.clamp(0.0, 1.0) if config.clip_output else pred


def psnr_ssim_np(img: np.ndarray, ref: np.ndarray) -> tuple[float, float]:
    def _t(arr: np.ndarray) -> Tensor:
        return torch.from_numpy(np.ascontiguousarray(arr)).float()[None, None]

    _img, _ref = _t(img), _t(ref)
    return float(calculate_psnr(_img, _ref).item()), float(calculate_ssim(_img, _ref).mean().item())


# 자기검증: 동일 영상은 PSNR 이 매우 커야 하고 SSIM 은 1 이어야 한다
_a = torch.rand(2, 1, 64, 64)
print("identical  psnr:", calculate_psnr(_a, _a).flatten().tolist(), " ssim:", calculate_ssim(_a, _a).flatten().tolist())
_b = (_a + 0.1).clamp(0, 1)
print("perturbed  psnr:", [round(v, 3) for v in calculate_psnr(_b, _a).flatten().tolist()])

## 4. Dataset

### 4-1. 데이터 생성 규칙 준수

> For each clean image, you may generate up to 6 dipole kernel orientations.
> For each orientation, you may generate up to 2 noisy images.
> **Do not exceed 12 corrupted images per clean image.**

`variant_mode="pool"` (기본) 은 `(파일명, variant_idx)` 로 노이즈를 **결정론적으로 확정**해
**noise 실현을 clean 1장당 12개로 고정**한다. epoch 을 몇 번 돌려도 이 수는 늘지 않는다.

| mode | clean 당 noise 실현 | 규칙 |
|---|---|---|
| `pool` (기본) | **12** (4 noise × 3 σ 실현) | 준수 |
| `onthefly` | epoch 수만큼 무한히 증가 | 조교 example 의 방식. 60 epoch 이면 상한의 5배 |

**flip augmentation 에 대한 주의**: `config.augment_flip=True` (기본) 이면 flip 4종 각각에 12개 실현이 붙으므로
서로 다른 corrupted 영상은 48종이 된다. flip 은 clean 영상 자체의 기하 변환이고
조교 example 도 동일하게 flip augmentation 을 쓰지만, 규칙을 가장 엄격하게 해석해야 한다면
`config.augment_flip = False` 로 두면 clean 당 corrupted 가 **정확히 12장**이 된다.

### 4-2. 층화(stratified) validation

`claude/iter3_review.md` 8절에서 무작위 추첨 validation 이 run 간 비교를 불가능하게 만든 것을 확인했다
(iter2 val 입력 25.88 dB vs iter3 val 29.31 dB — 서로 다른 문제를 풀고 있었다).

여기서는 `이미지 × noise 4종 × σ 3층` 을 **전수 열거**한다.

- 4종이 정확히 균등 → test 의 25/25/25/25 분포와 일치
- σ 는 범위를 3등분한 중앙값으로 고정 → 추첨 분산 0
- run 이 달라도 **정확히 같은 문제**를 푼다

In [ ]:
class DataKey(IntEnum):
    Label = 0
    Measure = 1
    Blur = 2
    NoisyLabel = 3
    NoiseId = 4
    Name = 5


def variant_spec(name: str, k: int) -> tuple[str, float, int]:
    # (파일명, variant index) -> (noise_type, sigma, seed). 결정론적이고 4종이 균등하다
    h = zlib.crc32(f"{name}|v{k}".encode("utf-8")) & 0x7FFFFFFF
    nz = NOISE_NAMES[k % len(NOISE_NAMES)]
    lo, hi = NOISE_RANGES[nz]
    return nz, random.Random(h).uniform(lo, hi), h


def stratified_val_spec(name: str, t: int, level: int, levels: int) -> tuple[str, float, int]:
    # noise 종류 t, sigma 층 level -> 결정론적 spec. sigma 는 범위의 (level+0.5)/levels 지점
    nz = NOISE_NAMES[t]
    lo, hi = NOISE_RANGES[nz]
    sigma = lo + (hi - lo) * (level + 0.5) / levels
    seed = zlib.crc32(f"{name}|val|{t}|{level}".encode("utf-8")) & 0x7FFFFFFF
    return nz, sigma, seed


def _flip(x: Tensor, code_: int) -> Tensor:
    if code_ & 1:
        x = torch.flip(x, dims=[-1])
    if code_ & 2:
        x = torch.flip(x, dims=[-2])
    return x


class TrainDataset(Dataset):
    def __init__(self, split: PackedSplit, variant_mode: str, variant_pool: int, augment: bool = True) -> None:
        self.split = split
        self.variant_mode = variant_mode
        self.variant_pool = variant_pool
        self.augment = augment

    def __len__(self) -> int:
        return len(self.split)

    def __getitem__(self, idx: int):
        name = self.split.names[idx]
        label = torch.from_numpy(self.split.get(idx)).float()

        if self.augment:
            label = _flip(label, random.randrange(4))

        if self.variant_mode == "pool":
            k = random.randrange(self.variant_pool)
            nz, sigma, seed = variant_spec(name, k)
        else:
            nz = random.choice(NOISE_NAMES)
            lo, hi = NOISE_RANGES[nz]
            sigma, seed = random.uniform(lo, hi), None

        blur, measure = degrade(label, nz, sigma, seed)

        # clean branch 용: dipole 없이 noise 만 얹은 영상 (독립적인 노이즈)
        nz2 = random.choice(NOISE_NAMES)
        lo2, hi2 = NOISE_RANGES[nz2]
        noisy_label = apply_noise(label, nz2, random.uniform(lo2, hi2), None)

        return (
            label[None],
            measure[None],
            blur[None],
            noisy_label[None],
            NOISE_NAMES.index(nz),
            name,
        )


class StratifiedValDataset(Dataset):
    def __init__(self, split: PackedSplit, levels: int) -> None:
        self.split = split
        self.levels = levels
        self.combos = [(i, t, l) for i in range(len(split)) for t in range(len(NOISE_NAMES)) for l in range(levels)]

    def __len__(self) -> int:
        return len(self.combos)

    def __getitem__(self, idx: int):
        i, t, l = self.combos[idx]
        name = self.split.names[i]
        label = torch.from_numpy(self.split.get(i)).float()
        nz, sigma, seed = stratified_val_spec(name, t, l, self.levels)
        blur, measure = degrade(label, nz, sigma, seed)
        return label[None], measure[None], blur[None], label[None], t, name


class TestDataset(Dataset):
    # 제공된 test_deconv_noise(measure) + test_label(정답)
    def __init__(self) -> None:
        self.measure = PackedSplit("test_measure")
        self.label = PackedSplit("test_label")
        pos = self.label.name_to_pos()
        missing = [n for n in self.measure.names if n not in pos]
        if missing:
            raise KeyError(f"test_label 에 없는 파일: {missing[:3]}")
        self.label_pos = [pos[n] for n in self.measure.names]

    def __len__(self) -> int:
        return len(self.measure)

    def __getitem__(self, idx: int):
        name = self.measure.names[idx]
        measure = torch.from_numpy(self.measure.get(idx)).float()
        label = torch.from_numpy(self.label.get(self.label_pos[idx])).float()
        blur = dipole_forward(label)
        nz = TEST_NOISE_META.get(name, {}).get("noise_type")
        nid = NOISE_NAMES.index(nz) if nz in NOISE_NAMES else -1
        return label[None], measure[None], blur[None], label[None], nid, name


def make_loaders() -> tuple[DataLoader, DataLoader, DataLoader]:
    train_split = PackedSplit(config.train_split, limit=config.max_train_images)
    val_split = PackedSplit(config.valid_split, l1_only=config.valid_l1_only)

    train_ds = TrainDataset(train_split, config.variant_mode, config.variant_pool, augment=config.augment_flip)
    val_ds = StratifiedValDataset(val_split, config.val_sigma_levels)
    test_ds = TestDataset()
    return train_ds, val_ds, test_ds


train_ds, val_ds, test_ds = make_loaders()
test_dataset = test_ds  # example 호환 별칭
print(f"train : {len(train_ds)}장 | variant_mode={config.variant_mode} pool={config.variant_pool} flip={config.augment_flip}")
_per_clean = config.variant_pool * (4 if config.augment_flip else 1) if config.variant_mode == "pool" else -1
print(f"        → clean 당 noise 실현 {config.variant_pool if config.variant_mode == 'pool' else '무제한(규칙 위반 위험)'}"
      f" | flip 포함 corrupted 종수 {_per_clean if _per_clean > 0 else '무제한'}")
print(f"val   : 이미지 {len(val_ds.split)}장 × noise {len(NOISE_NAMES)}종 × σ {config.val_sigma_levels}층 = {len(val_ds)} 조합")
print(f"test  : {len(test_ds)}장")

print()
print("variant pool 자기검증 (같은 clean 의 12개 variant):")
_nm = train_ds.split.names[0]
for _k in range(config.variant_pool):
    _nz, _sg, _sd = variant_spec(_nm, _k)
    print(f"  v{_k:<3}{_nz:<18}sigma={_sg:.4f}")
_a = variant_spec(_nm, 0)
print(f"  재현성: variant_spec 재호출 동일 → {_a == variant_spec(_nm, 0)}")

### 4-3. 학습 pair 눈으로 확인

`label x` → `dipole blur Ax` → `measure N(Ax)` 가 실제로 만들어지는지,
그리고 clean branch 입력 `N(x)` 가 어떻게 보이는지 확인한다.

In [ ]:
_loader = DataLoader(train_ds, batch_size=6, shuffle=True, num_workers=0)
_b = next(iter(_loader))
_l, _m, _bl, _nl, _nid, _n = _b
print(f"label   : {tuple(_l.shape)} range [{_l.min():.3f}, {_l.max():.3f}]")
print(f"blur    : {tuple(_bl.shape)} range [{_bl.min():.3f}, {_bl.max():.3f}]")
print(f"measure : {tuple(_m.shape)} range [{_m.min():.3f}, {_m.max():.3f}]")
print(f"noise   : {[NOISE_NAMES[i] for i in _nid.tolist()]}")

n = _l.shape[0]
fig, axes = plt.subplots(4, n, figsize=(2.6 * n, 11))
for i in range(n):
    axes[0, i].imshow(_l[i, 0].numpy(), cmap="gray", vmin=0, vmax=1)
    axes[0, i].set_title(f"label x\n{_n[i][:14]}", fontsize=8)
    _bb = _bl[i, 0].numpy()
    axes[1, i].imshow(_bb, cmap="gray", vmin=np.percentile(_bb, 1), vmax=np.percentile(_bb, 99))
    axes[1, i].set_title("dipole blur Ax", fontsize=8)
    _mm = _m[i, 0].numpy()
    axes[2, i].imshow(_mm, cmap="gray", vmin=np.percentile(_mm, 1), vmax=np.percentile(_mm, 99))
    axes[2, i].set_title(f"measure N(Ax)\n{NOISE_NAMES[_nid[i]]}", fontsize=8)
    axes[3, i].imshow(_nl[i, 0].numpy(), cmap="gray", vmin=0, vmax=1)
    axes[3, i].set_title("noisy label N(x)", fontsize=8)
    for r in range(4):
        axes[r, i].axis("off")
fig.suptitle("train synthetic pair  |  아래 두 줄이 denoiser 가 보는 두 종류 입력")
fig.tight_layout(rect=(0, 0, 1, 0.95))
plt.show()
del _loader

## 5. Conventional methods

학습 없이 forward model 을 안다는 가정만으로 복원한다.

1. **denoise** : mean / median / adaptive filter
2. **deconvolve** : Wiener filter $\hat{x}=\mathcal{F}^{-1}\{\frac{D}{D^2+K}\mathcal{F}\{y\}\}$

`D` 가 실수이므로 `(1/D)·D²/(D²+K) = D/(D²+K)` 로 0 나눗셈 없이 계산된다.
즉 **Wiener 와 Tikhonov 는 이 문제에서 동일한 필터**다.

`K` 는 validation 에서 PSNR 이 최대가 되도록 sweep 해서 고른다 (label 을 쓰는 튜닝이므로 test 는 건드리지 않는다).

In [ ]:
def _as_bchw(img: Tensor) -> tuple[Tensor, int]:
    dim = img.dim()
    if dim == 2:
        return img[None, None], dim
    if dim == 3:
        return img[None], dim
    if dim == 4:
        return img, dim
    raise ValueError(f"unsupported image dim: {dim}")


def _restore_dim(img: Tensor, dim: int) -> Tensor:
    if dim == 2:
        return img[0, 0]
    if dim == 3:
        return img[0]
    return img


def mean_filter(img: Tensor, kernel_size: int = 3) -> Tensor:
    x, dim = _as_bchw(img)
    pad = kernel_size // 2
    x = functional.pad(x, (pad, pad, pad, pad), mode="reflect")
    return _restore_dim(functional.avg_pool2d(x, kernel_size=kernel_size, stride=1), dim)


def median_filter(img: Tensor, kernel_size: int = 3) -> Tensor:
    x, dim = _as_bchw(img)
    pad = kernel_size // 2
    x = functional.pad(x, (pad, pad, pad, pad), mode="reflect")
    patches = x.unfold(2, kernel_size, 1).unfold(3, kernel_size, 1)
    out = patches.reshape(*patches.shape[:4], -1).median(dim=-1).values
    return _restore_dim(out, dim)


def adaptive_filter(img: Tensor, kernel_size: int = 5, noise_var: Tensor | float | None = None) -> Tensor:
    x, dim = _as_bchw(img)
    pad = kernel_size // 2
    xp = functional.pad(x, (pad, pad, pad, pad), mode="reflect")

    local_mean = functional.avg_pool2d(xp, kernel_size=kernel_size, stride=1)
    local_sq = functional.avg_pool2d(xp.pow(2), kernel_size=kernel_size, stride=1)
    local_var = (local_sq - local_mean.pow(2)).clamp_min(0.0)

    if noise_var is None:
        noise_var = local_var.flatten(2).median(dim=-1).values[:, :, None, None]

    ratio = (noise_var / local_var.clamp_min(1e-8)).clamp(max=1.0)
    return _restore_dim(x - ratio * (x - local_mean), dim)


def wiener_deconv(img: Tensor, K: float) -> Tensor:
    kernel = dipole_kernel(tuple(img.shape[-2:])).to(img.device)
    w = kernel / (kernel**2 + K)
    img_k = torch.fft.fftn(img.float(), dim=(-2, -1))
    return torch.fft.ifftn(img_k * w, dim=(-2, -1)).real


def tkd(img: Tensor, clip: float = 5.0) -> Tensor:
    kernel = dipole_kernel(tuple(img.shape[-2:])).to(img.device)
    kernel_inv = torch.clip(1 / kernel, min=-clip, max=clip)
    img_k = torch.fft.fftn(img.float(), dim=(-2, -1))
    return torch.fft.ifftn(img_k * kernel_inv, dim=(-2, -1)).real


BASELINE_KERNEL: int = 3
ADAPTIVE_KERNEL: int = 5

PREFILTERS: dict[str, Callable[[Tensor], Tensor]] = {
    "none": lambda x: x,
    "mean": lambda x: mean_filter(x, kernel_size=BASELINE_KERNEL),
    "median": lambda x: median_filter(x, kernel_size=BASELINE_KERNEL),
    "adaptive": lambda x: adaptive_filter(x, kernel_size=ADAPTIVE_KERNEL),
}
PREFILTER_LABEL: dict[str, str] = {
    "none": "Wiener only",
    "mean": f"Mean {BASELINE_KERNEL}x{BASELINE_KERNEL} + Wiener",
    "median": f"Median {BASELINE_KERNEL}x{BASELINE_KERNEL} + Wiener",
    "adaptive": f"Adaptive {ADAPTIVE_KERNEL}x{ADAPTIVE_KERNEL} + Wiener",
}

# Wiener == Tikhonov 항등식 검증
_t = torch.rand(1, 1, 64, 64)
_k = dipole_kernel((64, 64))
_w1 = _k / (_k**2 + 1e-3)
_w2 = (1 / _k) * (_k**2 / (_k**2 + 1e-3))
print("Wiener == Tikhonov:", torch.allclose(_w1, _w2, atol=1e-5), f"(max diff {float((_w1 - _w2).abs().max()):.2e})")

In [ ]:
WIENER_K_GRID: list[float] = [float(k) for k in np.logspace(-4, 0.5, 19)]

valid_loader = DataLoader(val_ds, batch_size=config.valid_batch, shuffle=False, num_workers=config.num_workers)
test_loader = DataLoader(test_ds, batch_size=config.valid_batch, shuffle=False, num_workers=config.num_workers)


def sweep_wiener_K(
    loader: DataLoader,
    pre: Callable[[Tensor], Tensor],
    k_grid: list[float],
    device: torch.device,
) -> tuple[float, dict[float, float]]:
    acc: dict[float, list[float]] = {k: [] for k in k_grid}
    with torch.no_grad():
        for _data in loader:
            label = _data[DataKey.Label].to(device, non_blocking=True)
            measure = _data[DataKey.Measure].to(device, non_blocking=True)
            filtered = pre(measure)
            for k in k_grid:
                pred = finalize(wiener_deconv(filtered, K=k))
                acc[k].extend(calculate_psnr(pred, label).flatten().tolist())
    mean_psnr = {k: float(np.mean(v)) for k, v in acc.items()}
    return max(mean_psnr, key=mean_psnr.get), mean_psnr


BEST_K: dict[str, float] = {}
K_CURVES: dict[str, dict[float, float]] = {}

for _key, _fn in PREFILTERS.items():
    t0 = time.time()
    _best, _curve = sweep_wiener_K(valid_loader, _fn, WIENER_K_GRID, config.device)
    BEST_K[_key] = _best
    K_CURVES[_key] = _curve
    print(f"{PREFILTER_LABEL[_key]:<34} best K = {_best:.4g}  (valid PSNR {_curve[_best]:.3f} dB, {time.time() - t0:.1f}s)")

(WORK_ROOT / "best_k.json").write_text(json.dumps(BEST_K, indent=2), encoding="utf-8")

fig, ax = plt.subplots(figsize=(7.5, 4.6))
for _key in PREFILTERS:
    _curve = K_CURVES[_key]
    ax.semilogx(list(_curve.keys()), list(_curve.values()), marker="o", ms=3, label=PREFILTER_LABEL[_key])
    ax.scatter([BEST_K[_key]], [_curve[BEST_K[_key]]], s=70, zorder=5, facecolors="none", edgecolors="k")
ax.set_xlabel("Wiener K")
ax.set_ylabel("valid PSNR [dB]")
ax.set_title("Wiener K sweep on stratified validation")
ax.grid(alpha=0.3)
ax.legend(fontsize=9)
fig.tight_layout()
plt.show()

## 6. 노이즈 파워 실측 (iter2 설계 근거 재현)

`test_deconv_only`(= `Ax`, 노이즈 없음) 이 있으므로 test 의 실제 노이즈를 직접 분리할 수 있다.
헤더의 표를 이 셀이 재현한다. `test_deconv_only` 가 없으면 건너뛴다.

In [ ]:
if (CACHE_DIR / "test_blur.npy").exists() and TEST_NOISE_META:
    _tb = PackedSplit("test_blur")
    _tm2 = PackedSplit("test_measure")
    _pos_b = _tb.name_to_pos()
    stat: dict[str, list] = {}
    for _n in _tm2.names:
        _t = TEST_NOISE_META.get(_n, {}).get("noise_type")
        if _t not in NOISE_NAMES:
            continue
        _bl = _tb.get(_pos_b[_n])
        _me = _tm2.get(_tm2.name_to_pos()[_n])
        _r = (_me - _bl).astype(np.float64)
        stat.setdefault(_t, []).append((TEST_NOISE_META[_n]["sigma"], _r.std(), np.abs(_r).mean(), (_r**2).mean()))

    print("=== test set 노이즈 강도 실측 (n=25 per type) ===")
    print(f"{'type':<18}{'선언 sigma':>12}{'noise std':>12}{'E|noise|':>12}{'noise MSE':>12}")
    print("-" * 66)
    _ref = float(np.mean([x[3] for x in stat["gaussian"]]))
    for _t in NOISE_NAMES:
        _v = np.array(stat[_t])
        print(f"{_t:<18}{_v[:, 0].mean():>12.4f}{_v[:, 1].mean():>12.4f}{_v[:, 2].mean():>12.4f}{_v[:, 3].mean():>12.5f}")

    # iter1 C + flip TTA 실측치
    ITER1_TTA = {"gaussian": 28.636, "rician": 21.437, "uniform": 27.164, "salt_and_pepper": 28.819}
    print()
    print("=== 노이즈 파워로 설명되는 부분 vs 실측 격차 (기준: gaussian) ===")
    print(f"{'type':<18}{'MSE 비':>10}{'기대 손실':>12}{'iter1 실측':>12}{'미설명분':>12}")
    print("-" * 64)
    for _t in NOISE_NAMES:
        _mse = float(np.mean([x[3] for x in stat[_t]]))
        _exp = 10 * math.log10(_mse / _ref)
        _act = ITER1_TTA["gaussian"] - ITER1_TTA[_t]
        print(f"{_t:<18}{_mse / _ref:>10.3f}{_exp:>12.2f}{_act:>12.2f}{_act - _exp:>12.2f}")
    print()
    print("→ rician 격차의 대부분이 sigma 차이다. S&P 는 오히려 가장 쉬운 subset 이다.")
    print("→ 따라서 noise type 별 특수 처리보다 sigma 를 모델에 알려주는 것이 우선이다.")

    print()
    print("=== sigma 분포 ===")
    for _t in NOISE_NAMES:
        _v = np.sort(np.array(stat[_t])[:, 0])
        print(f"  {_t:<18} min {_v[0]:.4f} q25 {_v[6]:.4f} median {_v[12]:.4f} q75 {_v[18]:.4f} max {_v[-1]:.4f}")
    del _tb, _tm2
else:
    print("test_deconv_only 또는 noise_meta 가 없어 건너뛴다 (학습에는 영향 없음)")

## 7. σ̂ 추정 (label-free)

MAD(median absolute deviation) 기반. `median` 은 impulse 에 강건해서 salt & pepper 에서도 무너지지 않는다.

$$\hat{\sigma} = \frac{\mathrm{median}\big(|y - \mathrm{median}_{3\times3}(y)|\big)}{0.6745}$$

두 종류를 채널로 준다.

| 채널 | 내용 | 목적 |
|---|---|---|
| global σ̂ | 영상 전체 1개 값을 상수맵으로 | FFDNet 방식의 noise level map |
| local σ̂ | 16×16 블록별 σ̂ 를 upsample | 공간적으로 변하는 노이즈 대응 |

**주의**: dipole kernel `D(k) = 1/3 − k_y²/|k|²` 는 `k` 의 **방향에만 의존하고 크기와 무관**하다 (0차 동차).
즉 low-pass 가 아니라서 `Ax` 가 고주파를 그대로 보존하고, MAD 추정에 신호 성분이 섞인다.

실측(스모크)에서는 오히려 **약간 과소추정**되었다.

| noise | 선언 σ | σ̂ | 실제 noise std | σ̂ / 실제 std |
|---|---|---|---|---|
| gaussian | 0.0500 | 0.0453 | 0.0500 | 0.91 |
| rician | 0.0750 | 0.0589 | ≈0.0750 | 0.79 |
| uniform | 0.1000 | 0.0598 | 0.0577 (=σ/√3) | **1.04** |

uniform 에서 σ̂ 가 **실제 std 와 일치**한다는 점이 중요하다.
σ̂ 는 선언 σ 가 아니라 **실제 노이즈 표준편차**를 추정하고 있고, 이게 모델에 필요한 값이다.

절대값이 정확할 필요는 없다. **σ 의 순서(monotonicity)만 보존되면** 네트워크가 조건부로 쓸 수 있다.
아래 셀에서 Pearson/Spearman 상관을 측정해 이를 확인한다
(`val_sigma_levels=1` 이면 층이 하나뿐이라 상관이 정의되지 않으므로 3 이상이어야 한다).

In [ ]:
MAD_SCALE: float = 0.6745


def estimate_sigma_global(y: Tensor) -> Tensor:
    # (B,1,H,W) -> (B,1,1,1)
    med = median_filter(y, kernel_size=3)
    dev = (y - med).abs()
    mad = dev.flatten(2).median(dim=-1).values[:, :, None, None]
    return mad / MAD_SCALE


def estimate_sigma_local(y: Tensor, block: int = 16) -> Tensor:
    # 블록별 MAD -> 원래 크기로 upsample. (B,1,H,W)
    med = median_filter(y, kernel_size=3)
    dev = (y - med).abs()
    b, c, h, w = dev.shape
    hb, wb = h // block, w // block
    patches = dev[:, :, : hb * block, : wb * block].reshape(b, c, hb, block, wb, block)
    patches = patches.permute(0, 1, 2, 4, 3, 5).reshape(b, c, hb, wb, block * block)
    mad = patches.median(dim=-1).values / MAD_SCALE
    return functional.interpolate(mad, size=(h, w), mode="nearest")


# ---- 자기검증: 추정 sigma 가 실제 sigma 의 순서를 보존하는가 ----
_chk_split = PackedSplit(config.valid_split, l1_only=True)
_rows = []
for _t_idx, _t in enumerate(NOISE_NAMES):
    for _lv in range(config.val_sigma_levels):
        for _i in range(0, len(_chk_split), 7):
            _nm = _chk_split.names[_i]
            _lab = torch.from_numpy(_chk_split.get(_i)).float()
            _nz, _sg, _sd = stratified_val_spec(_nm, _t_idx, _lv, config.val_sigma_levels)
            _, _me = degrade(_lab, _nz, _sg, _sd)
            _g = float(estimate_sigma_global(_me[None, None]).item())
            _rows.append((_nz, _sg, _g))

print(f"{'noise':<18}{'true sigma':>12}{'sigma_hat':>12}{'비율':>10}")
print("-" * 52)
for _t in NOISE_NAMES:
    _v = [(s, g) for n, s, g in _rows if n == _t]
    _s = np.array([x[0] for x in _v])
    _g = np.array([x[1] for x in _v])
    _corr = float(np.corrcoef(_s, _g)[0, 1])
    print(f"{_t:<18}{_s.mean():>12.4f}{_g.mean():>12.4f}{(_g / np.maximum(_s, 1e-6)).mean():>10.2f}   corr={_corr:+.3f}")

_s_all = np.array([r[1] for r in _rows])
_g_all = np.array([r[2] for r in _rows])
print()
print(f"전체 상관계수 (Pearson) : {float(np.corrcoef(_s_all, _g_all)[0, 1]):+.4f}")
from scipy.stats import spearmanr

print(f"전체 순위상관 (Spearman): {float(spearmanr(_s_all, _g_all).statistic):+.4f}")
print("→ 순위상관이 높으면 절대값이 부정확해도 조건부 입력으로 쓸 수 있다.")

fig, ax = plt.subplots(figsize=(5.6, 4.4))
for _t in NOISE_NAMES:
    _v = [(s, g) for n, s, g in _rows if n == _t]
    ax.scatter([x[0] for x in _v], [x[1] for x in _v], s=16, alpha=0.7, label=_t)
ax.plot([0, 0.2], [0, 0.2], "k--", lw=1, label="y = x")
ax.set_xlabel("true sigma")
ax.set_ylabel("estimated sigma (MAD)")
ax.set_title("sigma estimation on stratified val")
ax.grid(alpha=0.3)
ax.legend(fontsize=8)
fig.tight_layout()
plt.show()
del _chk_split

## 8. Models

iter1 의 `Unet` / `DnCNN` / `PhysResUnet` 을 **그대로 유지**한다 (iter1 체크포인트 로드에 필요).
여기에 `PhysResUnetSig` 를 추가한다.

```
measure ─┬──────────────────────────────► ch0
         ├─ Wiener(measure, K)      ────► ch1   (= base, residual 기준)
         ├─ Wiener(measure, K×10)   ────► ch2
         ├─ global σ̂ 상수맵          ────► ch3   ← iter2 추가
         └─ local σ̂ (16×16 블록)     ────► ch4   ← iter2 추가
                        │
                        ▼  5채널
                     U-Net (chans 48)
                        │
   output = base + scale · UNet(...)     ← 마지막 conv zero-init
```

zero-init 은 유지한다. 학습 시작 시점 출력이 `Wiener(measure, K)` 와 정확히 같으므로
**conventional 성능이 하한으로 보장**된다.

In [ ]:
class ConvBlock(nn.Module):
    def __init__(self, in_chans: int, out_chans: int) -> None:
        super().__init__()
        self.layers = nn.Sequential(
            nn.Conv2d(in_chans, out_chans, kernel_size=3, padding=1),
            nn.GroupNorm(4, out_chans),
            nn.SiLU(inplace=True),
            nn.Conv2d(out_chans, out_chans, kernel_size=3, padding=1),
            nn.GroupNorm(4, out_chans),
            nn.SiLU(inplace=True),
        )

    def forward(self, x: Tensor) -> Tensor:
        return self.layers(x)


def create_down_sample_layers(in_chans: int, chans: int, num_pool_layers: int) -> nn.ModuleList:
    layers = nn.ModuleList([ConvBlock(in_chans, chans)])
    ch = chans
    for _ in range(num_pool_layers - 1):
        layers.append(ConvBlock(ch, ch * 2))
        ch *= 2
    return layers


def create_up_sample_layers(chans: int, num_pool_layers: int) -> nn.ModuleList:
    layers = nn.ModuleList()
    ch = chans * (2 ** (num_pool_layers - 1))
    for _ in range(num_pool_layers - 1):
        layers.append(ConvBlock(ch * 2, ch // 2))
        ch //= 2
    layers.append(ConvBlock(ch * 2, ch))
    return layers


class Unet(nn.Module):
    def __init__(self, in_chans: int = 1, out_chans: int = 1, chans: int = 32, num_pool_layers: int = 4) -> None:
        super().__init__()
        self.in_chans = in_chans
        self.out_chans = out_chans
        self.num_pool_layers = num_pool_layers

        self.down_sample_layers = create_down_sample_layers(in_chans, chans, num_pool_layers)
        self.bottleneck_conv = ConvBlock(chans * (2 ** (num_pool_layers - 1)), chans * (2 ** (num_pool_layers - 1)))
        self.up_sample_layers = create_up_sample_layers(chans, num_pool_layers)
        self.final_conv = nn.Sequential(
            nn.Conv2d(chans, chans, kernel_size=3, padding=1),
            nn.GroupNorm(4, chans),
            nn.SiLU(inplace=True),
            nn.Conv2d(chans, out_chans, kernel_size=1, padding=0),
        )

    def forward(self, x: Tensor) -> Tensor:
        stack = []
        output = x
        for layer in self.down_sample_layers:
            output = layer(output)
            stack.append(output)
            output = functional.max_pool2d(output, kernel_size=2)

        output = self.bottleneck_conv(output)

        for layer in self.up_sample_layers:
            downsampled_output = stack.pop()
            output = functional.interpolate(
                output, size=downsampled_output.shape[-2:], mode="bilinear", align_corners=False
            )
            output = torch.cat([output, downsampled_output], dim=1)
            output = layer(output)

        return self.final_conv(output)


class DnCNN(nn.Module):
    def __init__(self, channels: int, num_of_layers: int, kernel_size: int, padding: int, features: int) -> None:
        super().__init__()
        layers: list[nn.Module] = [
            nn.Conv2d(channels, features, kernel_size=kernel_size, padding=padding, bias=False),
            nn.SiLU(inplace=True),
        ]
        for _ in range(num_of_layers - 1):
            layers += [
                nn.Conv2d(features, features, kernel_size=kernel_size, padding=padding, bias=False),
                nn.GroupNorm(4, features),
                nn.SiLU(inplace=True),
            ]
        layers.append(nn.Conv2d(features, channels, kernel_size=kernel_size, padding=padding, bias=False))
        self.dncnn = nn.Sequential(*layers)

    def forward(self, x: Tensor) -> Tensor:
        if x.dim() != 4:
            raise ValueError(f"Input tensor must be 4D, but got {x.dim()}D tensor.")
        return x + self.dncnn(x)


def _zero_init_last(net: Unet) -> None:
    last = net.final_conv[-1]
    nn.init.zeros_(last.weight)
    if last.bias is not None:
        nn.init.zeros_(last.bias)


class PhysResUnet(nn.Module):
    # iter1 의 C. 체크포인트 로드를 위해 그대로 유지한다
    def __init__(self, chans: int = 48, num_pool_layers: int = 4, wiener_K: float = 0.01, scale: float = 0.1) -> None:
        super().__init__()
        self.wiener_K = wiener_K
        self.scale = scale
        self.net = Unet(in_chans=3, out_chans=1, chans=chans, num_pool_layers=num_pool_layers)
        _zero_init_last(self.net)

    def features(self, measure: Tensor) -> tuple[Tensor, Tensor]:
        with torch.autocast(device_type=measure.device.type, enabled=False):
            m = measure.float()
            base = wiener_deconv(m, K=self.wiener_K)
            soft = wiener_deconv(m, K=self.wiener_K * 10.0)
            x = torch.cat([m, base, soft], dim=1)
        return x, base

    def forward(self, measure: Tensor) -> Tensor:
        x, base = self.features(measure)
        return base + self.scale * self.net(x)


class PhysResUnetSig(nn.Module):
    # iter2. PhysResUnet + global/local sigma_hat 채널
    def __init__(
        self,
        chans: int = 48,
        num_pool_layers: int = 4,
        wiener_K: float = 0.01,
        scale: float = 0.1,
        sigma_gain: float = 10.0,
        local_block: int = 16,
    ) -> None:
        super().__init__()
        self.wiener_K = wiener_K
        self.scale = scale
        self.sigma_gain = sigma_gain
        self.local_block = local_block
        self.net = Unet(in_chans=5, out_chans=1, chans=chans, num_pool_layers=num_pool_layers)
        _zero_init_last(self.net)

    def features(self, measure: Tensor) -> tuple[Tensor, Tensor]:
        with torch.autocast(device_type=measure.device.type, enabled=False):
            m = measure.float()
            base = wiener_deconv(m, K=self.wiener_K)
            soft = wiener_deconv(m, K=self.wiener_K * 10.0)
            sg = estimate_sigma_global(m).expand_as(m) * self.sigma_gain
            sl = estimate_sigma_local(m, block=self.local_block) * self.sigma_gain
            x = torch.cat([m, base, soft, sg, sl], dim=1)
        return x, base

    def forward(self, measure: Tensor) -> Tensor:
        x, base = self.features(measure)
        return base + self.scale * self.net(x)


MODEL_CONFIG_DEFAULT = {
    "in_chans": 1,
    "out_chans": 1,
    "chans": 64,
    "num_pool_layers": 4,
    "dncnn_layers": 20,
    "dncnn_features": 96,
    "dncnn_kernel": 3,
    "phys_chans": 48,
    "phys_scale": 0.1,
    "sigma_gain": 10.0,
    "local_block": 16,
}


def build_model(spec: dict) -> nn.Module:
    mc = {**MODEL_CONFIG_DEFAULT, **spec.get("model_config", {})}
    arch = spec["arch"]
    if arch == "unet":
        return Unet(mc["in_chans"], mc["out_chans"], mc["chans"], mc["num_pool_layers"])
    if arch == "dncnn":
        return DnCNN(1, mc["dncnn_layers"], mc["dncnn_kernel"], mc["dncnn_kernel"] // 2, mc["dncnn_features"])
    if arch == "phys_unet":
        return PhysResUnet(mc["phys_chans"], mc["num_pool_layers"], spec.get("wiener_K", 0.01), mc["phys_scale"])
    if arch == "phys_unet_sig":
        return PhysResUnetSig(
            mc["phys_chans"],
            mc["num_pool_layers"],
            spec.get("wiener_K", 0.01),
            mc["phys_scale"],
            mc["sigma_gain"],
            mc["local_block"],
        )
    raise ValueError(f"unknown arch: {arch}")


for _k, _s in [
    ("unet(64)", {"arch": "unet"}),
    ("dncnn(20,96)", {"arch": "dncnn"}),
    ("phys_unet(48) iter1", {"arch": "phys_unet"}),
    ("phys_unet_sig(48) iter2", {"arch": "phys_unet_sig"}),
]:
    print(f"{_k:<26}{sum(p.numel() for p in build_model(_s).parameters()):>12,} params")

# zero-init 검증: 초기 출력이 Wiener base 와 같아야 한다
_m0 = build_model({"arch": "phys_unet_sig", "wiener_K": BEST_K["none"]}).to(config.device).eval()
_x0 = torch.rand(2, 1, 256, 256, device=config.device) * 0.4 - 0.1
with torch.no_grad():
    _out0 = _m0(_x0)
    _base0 = wiener_deconv(_x0, K=BEST_K["none"])
print(f"\nzero-init 확인: max|out - Wiener_base| = {float((_out0 - _base0).abs().max()):.3e}  (0 이어야 정상)")
del _m0, _x0, _out0, _base0

## 8-2. iter3 추가 입력 신호

σ̂ 외에 두 가지를 더 준다. 둘 다 **label 을 쓰지 않는다.**

| 신호 | 정의 | 무엇을 알려주는가 |
|---|---|---|
| `frac_neg` | 음수 픽셀 비율 (상수맵) | **rician 판별**. rician 은 `\|Ax+n\|` 이라 0, 나머지는 > 0 |
| `impulse` | `clamp(\|y − median₃(y)\| / (σ̂+ε), 0, 8) / 8` | **S&P 를 공간적으로 국소화** |

`frac_neg` 의 판별력은 실측으로 확인되었다.

| noise | 측정값 min ≥ 0 인 장수 | 판정 |
|---|---|---|
| rician | **25 / 25** | 항상 비음수 |
| gaussian | 0 / 25 | 항상 음수 포함 |
| uniform | 4 / 25 | 대개 음수 포함 |
| salt_and_pepper | 7 / 25 | 대개 음수 포함 |

In [ ]:
def frac_negative(y: Tensor) -> Tensor:
    # (B,1,H,W) -> (B,1,1,1). rician 이면 0 에 가깝다
    return (y < 0).float().mean(dim=(-2, -1), keepdim=True)


def impulse_map(y: Tensor, sigma: Tensor, cap: float = 8.0) -> Tensor:
    # median 편차를 sigma 로 정규화. S&P 의 impulse 위치가 1 에 가까워진다
    med = median_filter(y, kernel_size=3)
    dev = (y - med).abs() / (sigma + 1e-6)
    return dev.clamp(0.0, cap) / cap


# ---- 자기검증: frac_neg 가 noise type 을 가르는가 ----
_chk = PackedSplit(config.valid_split, l1_only=True)
_res: dict[str, list] = {}
for _t_idx, _t in enumerate(NOISE_NAMES):
    for _lv in range(config.val_sigma_levels):
        for _i in range(0, len(_chk), 5):
            _nm = _chk.names[_i]
            _lab = torch.from_numpy(_chk.get(_i)).float()
            _nz, _sg, _sd = stratified_val_spec(_nm, _t_idx, _lv, config.val_sigma_levels)
            _, _me = degrade(_lab, _nz, _sg, _sd)
            _m4 = _me[None, None]
            _s = estimate_sigma_global(_m4)
            _res.setdefault(_t, []).append(
                (float(frac_negative(_m4).item()), float(impulse_map(_m4, _s).mean().item()), float(_s.item()))
            )

print(f"{'noise':<18}{'frac_neg 평균':>14}{'frac_neg=0 비율':>17}{'impulse 평균':>14}{'sigma_hat':>11}")
print("-" * 76)
for _t in NOISE_NAMES:
    _v = np.array(_res[_t])
    print(
        f"{_t:<18}{_v[:, 0].mean():>14.4f}{(_v[:, 0] < 1e-6).mean() * 100:>16.1f}%"
        f"{_v[:, 1].mean():>14.4f}{_v[:, 2].mean():>11.4f}"
    )
print()
print("→ rician 의 frac_neg 가 0 이고 나머지는 0 보다 크면 판별 신호로 작동한다.")
print("→ impulse 평균이 salt_and_pepper 에서 가장 크면 국소화 신호로 작동한다.")
del _chk

## 8-3. Unrolled HQS 모델

```
x₀ = Wiener(y, K)
반복 t = 1..T:
    z = x + denoiser([x, y, σ̂, frac_neg, impulse])     ← residual, zero-init
    X = (D·Y + λ_t·Z) / (D² + λ_t)                     ← 닫힌 해
    x = ifft(X).real
```

- denoiser 는 **가중치 공유**(기본). `share_denoiser=False` 로 반복별 분리 가능
- `λ_t = exp(θ_t)`, `θ` 는 학습 파라미터, `λ_init = 10.0`
- FFT 연산은 전부 `autocast(enabled=False)` 로 fp32 강제
- 중간 출력을 `_LAST_INTER` 에 저장해 deep supervision 에 사용

In [ ]:
_LAST_INTER: list[Tensor] = []


class UnrolledHQS(nn.Module):
    def __init__(
        self,
        chans: int = 32,
        num_pool_layers: int = 4,
        iters: int = 4,
        wiener_K: float = 0.01,
        lam_init: float = 10.0,
        share_denoiser: bool = True,
        sigma_gain: float = 10.0,
    ) -> None:
        super().__init__()
        self.iters = iters
        self.wiener_K = wiener_K
        self.share = share_denoiser
        self.sigma_gain = sigma_gain
        n_net = 1 if share_denoiser else iters
        # 입력 5채널: [x, y, sigma_hat, frac_neg, impulse]
        self.denoisers = nn.ModuleList(
            [Unet(in_chans=5, out_chans=1, chans=chans, num_pool_layers=num_pool_layers) for _ in range(n_net)]
        )
        for d in self.denoisers:
            _zero_init_last(d)
        self.log_lam = nn.Parameter(torch.full((iters,), math.log(lam_init)))

    def _side(self, measure: Tensor) -> tuple[Tensor, Tensor, Tensor, Tensor]:
        # 반복 내에서 변하지 않는 부가 입력을 한 번만 만든다
        with torch.autocast(device_type=measure.device.type, enabled=False):
            m = measure.float()
            sg = estimate_sigma_global(m)
            sig_map = sg.expand_as(m) * self.sigma_gain
            neg = frac_negative(m).expand_as(m)
            imp = impulse_map(m, sg)
        return m, sig_map, neg, imp

    def forward(self, measure: Tensor) -> Tensor:
        global _LAST_INTER
        m, sig_map, neg, imp = self._side(measure)

        with torch.autocast(device_type=measure.device.type, enabled=False):
            kernel = dipole_kernel(tuple(m.shape[-2:])).to(m.device)
            Y = torch.fft.fftn(m, dim=(-2, -1))
            DY = kernel * Y
            D2 = kernel**2
            x = wiener_deconv(m, K=self.wiener_K)

        inter: list[Tensor] = []
        for t in range(self.iters):
            net = self.denoisers[0 if self.share else t]
            z = x + net(torch.cat([x, m, sig_map, neg, imp], dim=1))
            with torch.autocast(device_type=measure.device.type, enabled=False):
                lam = self.log_lam[t].exp().float()
                Z = torch.fft.fftn(z.float(), dim=(-2, -1))
                x = torch.fft.ifftn((DY + lam * Z) / (D2 + lam), dim=(-2, -1)).real
            inter.append(x)

        _LAST_INTER = inter[:-1]  # 마지막은 pred 로 따로 감독한다
        return x


class SobelLoss(nn.Module):
    def __init__(self) -> None:
        super().__init__()
        kx = torch.tensor([[-1.0, 0.0, 1.0], [-2.0, 0.0, 2.0], [-1.0, 0.0, 1.0]])
        self.register_buffer("kx", kx[None, None])
        self.register_buffer("ky", kx.t().contiguous()[None, None])

    def forward(self, a: Tensor, b: Tensor) -> Tensor:
        kx, ky = self.kx.to(a.device), self.ky.to(a.device)
        gxa, gya = functional.conv2d(a, kx, padding=1), functional.conv2d(a, ky, padding=1)
        gxb, gyb = functional.conv2d(b, kx, padding=1), functional.conv2d(b, ky, padding=1)
        return functional.l1_loss(gxa, gxb) + functional.l1_loss(gya, gyb)


sobel_loss = SobelLoss()

MODEL_CONFIG_DEFAULT = {
    **MODEL_CONFIG_DEFAULT,
    "unroll_iters": 4,
    "unroll_chans": 32,
    "lam_init": 10.0,
    "share_denoiser": True,
}

_build_model_prev = build_model


def build_model(spec: dict) -> nn.Module:
    if spec["arch"] != "unrolled":
        return _build_model_prev(spec)
    mc = {**MODEL_CONFIG_DEFAULT, **spec.get("model_config", {})}
    return UnrolledHQS(
        chans=mc["unroll_chans"],
        num_pool_layers=mc["num_pool_layers"],
        iters=mc["unroll_iters"],
        wiener_K=spec.get("wiener_K", 0.01),
        lam_init=mc["lam_init"],
        share_denoiser=mc["share_denoiser"],
        sigma_gain=mc["sigma_gain"],
    )


_spec_probe = {"arch": "unrolled", "wiener_K": BEST_K["none"], "model_config": {}}
_mu = build_model(_spec_probe)
print(f"UnrolledHQS(iters=4, chans=32, share=True): {sum(p.numel() for p in _mu.parameters()):,} params")
print(f"  lambda 초기값: {_mu.log_lam.exp().tolist()}")

### 8-4. 하한 검증 — 학습 전 출력이 Wiener 와 같은가

헤더의 수식 `ratio = 1 + K/(D² + λ)` 는 **k-space 주파수별 배율**이므로,
영상 영역에서 픽셀별 비율을 보면 주파수 성분이 섞여 그 값과 그대로 일치하지 않는다.

**실질 검증은 PSNR 이다.** 학습 전 출력이 `Wiener(K)` 와 같은 PSNR 을 내면
**conventional 성능이 하한으로 보장**된다.

In [ ]:
_mu = _mu.to(config.device).eval()
_vb = next(iter(DataLoader(val_ds, batch_size=8, shuffle=False, num_workers=0)))
_lab = _vb[DataKey.Label].to(config.device)
_mea = _vb[DataKey.Measure].to(config.device)

with torch.no_grad():
    _out = _mu(_mea)
    _wien = wiener_deconv(_mea, K=BEST_K["none"])

_ratio = (_out / (_wien + 1e-8)).flatten()
print(f"out / wiener  중앙값 {float(_ratio.median()):.5f}  (k-space 배율이 섞이므로 1 에 가깝기만 하면 된다)")
print(f"max|out - wiener|      {float((_out - _wien).abs().max()):.4e}")
print()
print(f"{'':<26}{'PSNR':>10}{'SSIM':>10}")
print("-" * 46)
_p_w = float(calculate_psnr(finalize(_wien), _lab).mean())
_p_o = float(calculate_psnr(finalize(_out), _lab).mean())
for _nm, _p in [("Wiener(K) 해석해", _wien), ("학습 전 UnrolledHQS", _out)]:
    print(f"{_nm:<26}{calculate_psnr(finalize(_p), _lab).mean():>10.3f}{calculate_ssim(finalize(_p), _lab).mean():>10.4f}")
print()
print(f"차이 {_p_o - _p_w:+.3f} dB  ->  {'하한 보장 성립' if abs(_p_o - _p_w) < 0.1 else '경고: lam_init 을 30 으로 올릴 것'}")
print(f"중간 반복 출력 개수 (deep supervision 대상): {len(_LAST_INTER)}")
del _mu, _vb, _lab, _mea, _out, _wien

## 9. 학습

iter1 과 동일한 루프를 쓴다. `dc_weight` 만 0 으로 내린다.

In [ ]:
def get_lr(epoch: int, spec: dict) -> float:
    lr, total = spec["lr"], spec["epochs"]
    warm = spec.get("warmup", 1)
    if spec.get("lr_schedule", "cosine") == "exp":
        return lr * spec.get("lr_decay", 0.95) ** max(0, epoch - spec.get("lr_tol", 1))
    if epoch <= warm:
        return lr * epoch / max(1, warm)
    t = (epoch - warm) / max(1, total - warm)
    mn = spec.get("lr_min_ratio", 0.02)
    return lr * (mn + (1 - mn) * 0.5 * (1 + math.cos(math.pi * t)))


def compute_loss(pred: Tensor, target: Tensor, measure: Tensor, spec: dict) -> Tensor:
    base = spec.get("loss_model", "l1")
    loss = functional.l1_loss(pred, target) if base == "l1" else functional.mse_loss(pred, target)
    w_ssim = spec.get("ssim_weight", 0.0)
    if w_ssim > 0:
        loss = loss + w_ssim * (1.0 - calculate_ssim(pred.clamp(0, 1), target.clamp(0, 1)).mean())
    w_dc = spec.get("dc_weight", 0.0)
    if w_dc > 0 and spec["target"] == "label":
        with torch.autocast(device_type=pred.device.type, enabled=False):
            loss = loss + w_dc * functional.l1_loss(dipole_forward(pred.float()), measure.float())
    return loss


def model_forward(model: nn.Module, measure: Tensor, spec: dict) -> Tensor:
    return model(measure)


@torch.no_grad()
def evaluate_loader(model: nn.Module, loader: DataLoader, spec: dict, deconv_K: float | None = None) -> dict:
    model.eval()
    per_type: dict[int, list[tuple[float, float]]] = {i: [] for i in range(len(NOISE_NAMES))}
    psnr_all, ssim_all = [], []
    for data in loader:
        label = data[DataKey.Label].to(config.device, non_blocking=True)
        measure = data[DataKey.Measure].to(config.device, non_blocking=True)
        nid = data[DataKey.NoiseId]
        pred = model_forward(model, measure, spec)
        if spec["target"] == "blur":
            pred = wiener_deconv(pred, K=deconv_K if deconv_K is not None else BEST_K["none"])
        pred = finalize(pred)
        p = calculate_psnr(pred, label).flatten().tolist()
        s = calculate_ssim(pred, label).flatten().tolist()
        psnr_all += p
        ssim_all += s
        for j, t in enumerate(nid.tolist()):
            if t >= 0:
                per_type[t].append((p[j], s[j]))
    out = {
        "psnr": float(np.mean(psnr_all)),
        "ssim": float(np.mean(ssim_all)),
        "psnr_std": float(np.std(psnr_all, ddof=1)) if len(psnr_all) > 1 else 0.0,
    }
    for t, v in per_type.items():
        if v:
            out[f"psnr_{NOISE_NAMES[t]}"] = float(np.mean([x[0] for x in v]))
            out[f"ssim_{NOISE_NAMES[t]}"] = float(np.mean([x[1] for x in v]))
    return out


def next_run_id(root: Path) -> int:
    root.mkdir(parents=True, exist_ok=True)
    ids = []
    for e in root.iterdir():
        if e.is_dir():
            try:
                ids.append(int(e.name.split("_")[0]))
            except ValueError:
                pass
    return max(ids) + 1 if ids else 0

In [ ]:
def train_model(spec: dict) -> dict:
    key = spec["key"]
    run_root = WORK_ROOT / "logs_final"
    run_dir = run_root / f"{next_run_id(run_root):05d}_{key}"
    run_dir.mkdir(parents=True, exist_ok=True)

    torch.manual_seed(config.seed)
    random.seed(config.seed)

    model = build_model(spec).to(config.device)
    n_param = sum(p.numel() for p in model.parameters())

    train_loader = DataLoader(
        train_ds,
        batch_size=spec["batch"],
        shuffle=True,
        num_workers=config.num_workers,
        pin_memory=torch.cuda.is_available(),
        drop_last=True,
        persistent_workers=config.num_workers > 0,
    )

    opt = AdamW(model.parameters(), lr=spec["lr"], weight_decay=spec.get("weight_decay", 1e-4))
    use_amp = config.amp and torch.cuda.is_available()
    scaler = torch.amp.GradScaler("cuda", enabled=use_amp)

    clean_p = spec.get("clean_branch_prob", 0.0)
    history: list[dict] = []
    best = {"psnr": -1e9, "epoch": -1}
    bad = 0
    t_start = time.time()

    print(f"=== {spec['name']} ({key}) ===")
    print(f"  params {n_param:,} | batch {spec['batch']} | epochs {spec['epochs']} | target {spec['target']}")
    print(f"  loss {spec.get('loss_model', 'l1')} | ssim_w {spec.get('ssim_weight', 0)} | dc_w {spec.get('dc_weight', 0)}")
    print(f"  clean_branch_prob {clean_p} | run_dir {run_dir.name}")

    for epoch in range(1, spec["epochs"] + 1):
        lr = get_lr(epoch, spec)
        for g in opt.param_groups:
            g["lr"] = lr

        model.train()
        tot, nb = 0.0, 0
        bar = tqdm(train_loader, desc=f"ep {epoch}/{spec['epochs']}", leave=False, unit="batch")
        for data in bar:
            label = data[DataKey.Label].to(config.device, non_blocking=True)
            measure = data[DataKey.Measure].to(config.device, non_blocking=True)
            blur = data[DataKey.Blur].to(config.device, non_blocking=True)
            noisy_label = data[DataKey.NoisyLabel].to(config.device, non_blocking=True)

            if clean_p > 0 and random.random() < clean_p:
                inp, tgt = noisy_label, label
            else:
                inp = measure
                tgt = label if spec["target"] == "label" else blur

            opt.zero_grad(set_to_none=True)
            with torch.autocast("cuda", enabled=use_amp):
                pred = model_forward(model, inp, spec)
                loss = compute_loss(pred, tgt, inp, spec)
            scaler.scale(loss).backward()
            if spec.get("grad_clip", 0) > 0:
                scaler.unscale_(opt)
                nn.utils.clip_grad_norm_(model.parameters(), spec["grad_clip"])
            scaler.step(opt)
            scaler.update()

            tot += float(loss.item())
            nb += 1
            bar.set_postfix(loss=f"{tot / nb:.4e}", lr=f"{lr:.2e}")

        rec = {"epoch": epoch, "lr": lr, "train_loss": tot / max(1, nb)}

        if epoch % spec.get("valid_interval", 1) == 0:
            val = evaluate_loader(model, valid_loader, spec)
            rec.update({f"val_{k}": v for k, v in val.items()})
            msg = " ".join(f"{NOISE_NAMES[i][:4]} {val.get(f'psnr_{NOISE_NAMES[i]}', float('nan')):.2f}" for i in range(4))
            print(
                f"  ep {epoch:>3} loss {rec['train_loss']:.4e} lr {lr:.2e} | "
                f"val PSNR {val['psnr']:.3f} SSIM {val['ssim']:.4f} | {msg}"
            )
            if val["psnr"] > best["psnr"]:
                best = {"psnr": val["psnr"], "ssim": val["ssim"], "epoch": epoch}
                bad = 0
                torch.save(
                    {
                        "model_state_dict": model.state_dict(),
                        "model_config": {**MODEL_CONFIG_DEFAULT, **spec.get("model_config", {})},
                        "spec": spec,
                        "best": best,
                    },
                    run_dir / "checkpoint_best.ckpt",
                )
            else:
                bad += 1
                if bad >= spec.get("early_stop_patience", 10):
                    print(f"  early stop at epoch {epoch} (best ep {best['epoch']})")
                    history.append(rec)
                    break

        history.append(rec)
        (run_dir / "history.json").write_text(json.dumps(history, indent=1), encoding="utf-8")

    elapsed = time.time() - t_start
    (run_dir / "history.json").write_text(json.dumps(history, indent=1), encoding="utf-8")
    print(f"  done: best val PSNR {best['psnr']:.3f} @ ep {best['epoch']} | {elapsed / 60:.1f} min")

    ckpt = torch.load(run_dir / "checkpoint_best.ckpt", map_location=config.device, weights_only=False)
    model.load_state_dict(ckpt["model_state_dict"])
    return {"key": key, "spec": spec, "model": model, "run_dir": run_dir, "best": best, "history": history, "params": n_param}

### 9-2. 손실 확장 (Sobel + deep supervision)

`compute_loss` 를 재정의한다. Python 은 호출 시점에 전역을 조회하므로
앞서 정의된 `train_model` 이 이 새 정의를 쓴다.

$$\mathcal{L} = \mathcal{L}_1 + 0.5(1-\text{SSIM}) + 0.1\,\mathcal{L}_{\text{Sobel}}
+ \frac{w_{ds}}{T-1}\sum_{t=1}^{T-1}\mathcal{L}_1(x_t, x^\*)$$

In [ ]:
def compute_loss(pred: Tensor, target: Tensor, measure: Tensor, spec: dict) -> Tensor:
    base = spec.get("loss_model", "l1")
    loss = functional.l1_loss(pred, target) if base == "l1" else functional.mse_loss(pred, target)

    w_ssim = spec.get("ssim_weight", 0.0)
    if w_ssim > 0:
        loss = loss + w_ssim * (1.0 - calculate_ssim(pred.clamp(0, 1), target.clamp(0, 1)).mean())

    w_sobel = spec.get("sobel_weight", 0.0)
    if w_sobel > 0:
        loss = loss + w_sobel * sobel_loss(pred.float(), target.float())

    w_ds = spec.get("deep_supervision", 0.0)
    if w_ds > 0 and _LAST_INTER:
        aux = sum(functional.l1_loss(z.float(), target.float()) for z in _LAST_INTER) / len(_LAST_INTER)
        loss = loss + w_ds * aux

    w_dc = spec.get("dc_weight", 0.0)
    if w_dc > 0 and spec["target"] == "label":
        with torch.autocast(device_type=pred.device.type, enabled=False):
            loss = loss + w_dc * functional.l1_loss(dipole_forward(pred.float()), measure.float())
    return loss


print("compute_loss 재정의 완료 (Sobel + deep supervision 지원)")
_a = torch.rand(2, 1, 64, 64, device=config.device)
_b = torch.rand(2, 1, 64, 64, device=config.device)
_sp = {"target": "label", "loss_model": "l1", "ssim_weight": 0.5, "sobel_weight": 0.1}
print(f"  smoke: loss = {float(compute_loss(_a, _b, _b, _sp)):.4f}")
del _a, _b

In [ ]:
SPECS: dict[str, dict] = {
    "unrolled": {
        "key": "unrolled",
        "name": "E. Unrolled HQS (4 iter)",
        "arch": "unrolled",
        "target": "label",
        "model_config": {
            "unroll_iters": 4,
            "unroll_chans": 32,
            "num_pool_layers": 4,
            "lam_init": 10.0,
            "share_denoiser": True,
            "sigma_gain": 10.0,
        },
        "wiener_K": BEST_K["none"],
        "clean_branch_prob": 0.0,
        "epochs": 40,
        "batch": 8,
        "lr": 3e-4,
        "loss_model": "l1",
        "ssim_weight": 0.5,
        "sobel_weight": 0.1,
        "deep_supervision": 0.3,
        "dc_weight": 0.0,
        "lr_schedule": "cosine",
        "early_stop_patience": 10,
        "grad_clip": 1.0,
    },
}

print(f"{'key':<12}{'arch':<12}{'params':>12}{'iters':>7}{'epochs':>8}{'batch':>7}")
print("-" * 60)
for _k, _s in SPECS.items():
    print(
        f"{_k:<12}{_s['arch']:<12}{sum(p.numel() for p in build_model(_s).parameters()):>12,}"
        f"{_s['model_config']['unroll_iters']:>7}{_s['epochs']:>8}{_s['batch']:>7}"
    )

RESULTS: dict[str, dict] = {}

print()
print("메모리 주의: 반복 4회분 activation 이 쌓인다. OOM 이면 batch 8 -> 4,")
print("             또는 model_config['unroll_chans'] 32 -> 24 로 낮춘다.")
print("시간 추정: iter1 C(단일 pass, chans48, batch16) 41초/epoch 실측 기준")
print("          chans32 x 4 반복 = 약 1.8배 -> 70~80초/epoch -> 40 epoch 약 50분")

# --- 스모크 테스트 (권장) ---
# config.max_train_images = 120
# train_ds, val_ds, test_ds = make_loaders()
# RESULTS["smoke"] = train_model({**SPECS["unrolled"], "epochs": 1})
# config.max_train_images = None
# train_ds, val_ds, test_ds = make_loaders()   # <-- 반드시 되돌릴 것

### 9-3. 학습 실행

**게이트**

| 시점 | 확인 | 실패 시 |
|---|---|---|
| 8-4 셀 | 학습 전 PSNR 이 Wiener 와 0.1 dB 이내 | `lam_init` 을 30 으로 올린다 |
| epoch 1 | val PSNR ≥ 21 (iter1 C 는 21.703) | `lr` 3e-4 → 1e-4, `deep_supervision` 0.3 → 0.1 |
| epoch 10 | val PSNR > 25 | `unroll_iters` 4 → 3, `unroll_chans` 32 → 48 |
| 전 구간 | `lambda` 가 감소하는지 (`log_lam.exp()`) | 감소하지 않으면 DC step 이 안 쓰이는 것 |

In [ ]:
RESULTS["unrolled"] = train_model(SPECS["unrolled"])

# 학습된 lambda 확인: 초기 10.0 에서 얼마나 내려갔는가
_lam = RESULTS["unrolled"]["model"].log_lam.exp().detach().cpu().tolist()
print()
print(f"학습된 lambda (초기 10.0): {[round(v, 4) for v in _lam]}")
print("→ 값이 내려갔으면 DC step 이 실제로 사용된 것이다 (prior 만 쓰는 게 아니다).")

## 10. Test 평가

`test_deconv_noise` 100장 입력 → `test_label` 기준 PSNR/SSIM.
`noise_meta.json` 은 결과 분석에만 쓴다.

**iter1 의 체크포인트를 재학습 없이 로드**해 같은 표에 넣는다.
`logs_final/*_phys_unet` (iter1 C) 과 `logs_final/*_unet_e2e` (iter1 A) 를 찾는다.

In [ ]:
# flip 등변성 재확인 (TTA 근거)
_x = torch.rand(1, 1, 256, 256)
for _name, _dims in [("lr", [-1]), ("ud", [-2]), ("both", [-2, -1])]:
    _a = dipole_forward(torch.flip(_x, dims=_dims))
    _b = torch.flip(dipole_forward(_x), dims=_dims)
    print(f"flip {_name:<5} max|A(Tx) - T(Ax)| = {float((_a - _b).abs().max()):.2e}")
_a = dipole_forward(torch.rot90(_x, 1, dims=(-2, -1)))
_b = torch.rot90(dipole_forward(_x), 1, dims=(-2, -1))
print(f"rot90       max|A(Tx) - T(Ax)| = {float((_a - _b).abs().max()):.2e}  <- TTA 불가")

In [ ]:
LOADED: dict[str, dict] = {}


def load_ckpt_model(path: Path, label: str) -> dict | None:
    if not path.exists():
        print(f"  [없음] {path}")
        return None
    ck = torch.load(path, map_location=config.device, weights_only=False)
    spec = ck.get("spec")
    if spec is None or "arch" not in spec:
        # 조교 제공 baseline: spec 에 arch 가 없다
        mc = ck["model_config"]
        model = Unet(mc["in_chans"], mc["out_chans"], mc["chans"], mc["num_pool_layers"])
        spec = {"target": "label", "name": label}
    else:
        model = build_model(spec)
    model.load_state_dict(ck["model_state_dict"])
    model = model.to(config.device).eval()
    n = sum(p.numel() for p in model.parameters())
    print(f"  [로드] {label:<34} {n:>12,} params  ({path.parent.name})")
    return {"model": model, "spec": spec, "params": n, "best": ck.get("best", {}), "label": label}


print("=== 제공 baseline ===")
_bl = load_ckpt_model(ROOT / "code_denoising+deconv" / "checkpoint_baseline_best.ckpt", "제공 baseline U-Net")
if _bl:
    LOADED["baseline_unet"] = _bl

print("=== iter1 체크포인트 ===")
_runs = sorted((WORK_ROOT / "logs_final").glob("*")) if (WORK_ROOT / "logs_final").is_dir() else []
for _suffix, _key, _label in [
    ("_phys_unet", "iter1_phys", "iter1 C. Phys-residual"),
    ("_unet_e2e", "iter1_unet", "iter1 A. End2End U-Net"),
]:
    _cand = [r for r in _runs if r.name.endswith(_suffix) and (r / "checkpoint_best.ckpt").exists()]
    if _cand:
        _r = load_ckpt_model(_cand[-1] / "checkpoint_best.ckpt", _label)
        if _r:
            LOADED[_key] = _r
    else:
        print(f"  [없음] *{_suffix}")

print()
print("로드된 모델:", list(LOADED.keys()))
print("이번에 학습한 모델:", list(RESULTS.keys()))

In [ ]:
@torch.no_grad()
def predict_model(model: nn.Module, spec: dict, measure: Tensor, deconv_K: float | None = None, tta: bool = False) -> Tensor:
    model.eval()

    def _once(x: Tensor) -> Tensor:
        out = model_forward(model, x, spec)
        if spec.get("target") == "blur":
            out = wiener_deconv(out, K=deconv_K if deconv_K is not None else BEST_K["none"])
        return out

    if not tta:
        return finalize(_once(measure))

    acc = torch.zeros_like(measure, dtype=torch.float32)
    for dims in ([], [-1], [-2], [-2, -1]):
        xin = torch.flip(measure, dims=dims) if dims else measure
        out = _once(xin)
        acc += torch.flip(out, dims=dims).float() if dims else out.float()
    return finalize(acc / 4.0)


def conventional_predict(measure: Tensor) -> dict[str, Tensor]:
    out = {"measure": finalize(measure)}
    for key, fn in PREFILTERS.items():
        out[f"conv_{key}"] = finalize(wiener_deconv(fn(measure), K=BEST_K[key]))
    return out


# 평가 대상 구성: conventional -> 로드된 모델 -> 이번 학습 -> TTA
METHODS: list[str] = ["measure", *[f"conv_{k}" for k in PREFILTERS]]
METHOD_LABEL: dict[str, str] = {"measure": "Measure (input)", **{f"conv_{k}": v for k, v in PREFILTER_LABEL.items()}}

for _k, _v in LOADED.items():
    METHODS.append(_k)
    METHOD_LABEL[_k] = _v["label"]
if "iter1_phys" in LOADED:
    METHODS.append("iter1_phys_tta")
    METHOD_LABEL["iter1_phys_tta"] = "iter1 C + flip TTA"
for _k in ("dncnn_blur", "unrolled"):
    if _k in RESULTS:
        METHODS.append(_k)
        METHOD_LABEL[_k] = RESULTS[_k]["spec"]["name"]
if "unrolled" in RESULTS:
    METHODS.append("unrolled_tta")
    METHOD_LABEL["unrolled_tta"] = "E. Unrolled HQS + flip TTA"


def all_predict(measure: Tensor) -> dict[str, Tensor]:
    out = conventional_predict(measure)
    for k, v in LOADED.items():
        out[k] = predict_model(v["model"], v["spec"], measure)
    if "iter1_phys" in LOADED:
        v = LOADED["iter1_phys"]
        out["iter1_phys_tta"] = predict_model(v["model"], v["spec"], measure, tta=True)
    for k in ("dncnn_blur", "unrolled"):
        if k in RESULTS:
            r = RESULTS[k]
            out[k] = predict_model(r["model"], r["spec"], measure, r.get("deconv_K"))
    if "unrolled" in RESULTS:
        r = RESULTS["unrolled"]
        out["unrolled_tta"] = predict_model(r["model"], r["spec"], measure, tta=True)
    return out


rows: list[dict] = []
with torch.no_grad():
    for _data in tqdm(test_loader, desc="test eval", unit="batch"):
        label = _data[DataKey.Label].to(config.device)
        measure = _data[DataKey.Measure].to(config.device)
        names = _data[DataKey.Name]
        preds = all_predict(measure)
        for i in range(label.shape[0]):
            lab_i = label[i : i + 1]
            row = {"file": names[i], "noise_type": TEST_NOISE_META.get(names[i], {}).get("noise_type", "unknown")}
            for key in METHODS:
                p = preds[key][i : i + 1]
                row[f"psnr_{key}"] = calculate_psnr(p, lab_i).item()
                row[f"ssim_{key}"] = calculate_ssim(p, lab_i).item()
            rows.append(row)
print(f"평가 완료: {len(rows)}장 × {len(METHODS)}방법")

In [ ]:
def print_metric_table(rows: list[dict], methods: list[str], labels: dict[str, str], width: int = 34) -> None:
    print(f"{'method':<{width}}{'n':>5}{'PSNR':>10}{'+-':>9}{'SSIM':>10}{'+-':>9}")
    print("-" * (width + 43))
    for key in methods:
        psnr = [r[f"psnr_{key}"] for r in rows]
        ssim = [r[f"ssim_{key}"] for r in rows]
        print(
            f"{labels[key]:<{width}}{len(rows):>5}"
            f"{np.mean(psnr):>10.3f}{np.std(psnr, ddof=1):>9.3f}"
            f"{np.mean(ssim):>10.4f}{np.std(ssim, ddof=1):>9.4f}"
        )


print("=== 전체 (test 100장) ===")
print_metric_table(rows, METHODS, METHOD_LABEL)

for _metric, _fmt in [("psnr", "11.3f"), ("ssim", "11.4f")]:
    print()
    print(f"=== noise type 별 {_metric.upper()} ===")
    _w = 34
    print(f"{'method':<{_w}}" + "".join(f"{n[:9]:>11}" for n in NOISE_NAMES) + f"{'ALL':>11}")
    print("-" * (_w + 55))
    for _key in METHODS:
        _line = f"{METHOD_LABEL[_key]:<{_w}}"
        for _nt in NOISE_NAMES:
            _v = [r[f"{_metric}_{_key}"] for r in rows if r["noise_type"] == _nt]
            _line += f"{np.mean(_v):>{_fmt}}" if _v else f"{'-':>11}"
        _line += f"{np.mean([r[f'{_metric}_{_key}'] for r in rows]):>{_fmt}}"
        print(_line)

(WORK_ROOT / "test_metrics_iter3.json").write_text(json.dumps(rows, indent=1, ensure_ascii=False), encoding="utf-8")
print(f"\n저장: {WORK_ROOT / 'test_metrics_iter3.json'}")

### 10-1. Paired 유의성 검정

`iter1_review.md` 5.4 에서 지적한 문제를 해결한다.
두 방법은 **같은 100장**을 쓰므로 올바른 검정은 **차이값의 표준오차**를 쓰는 paired 검정이다.
값의 표준편차(5.2 dB)를 쓰면 유의성을 과소평가한다.

In [ ]:
def paired_test(rows: list[dict], a: str, b: str, subset: str | None = None) -> dict:
    # a - b 의 paired 통계. subset 은 noise_type
    sel = [r for r in rows if subset is None or r["noise_type"] == subset]
    d = np.array([r[f"psnr_{a}"] - r[f"psnr_{b}"] for r in sel])
    n = len(d)
    se = float(np.std(d, ddof=1) / math.sqrt(n)) if n > 1 else float("nan")
    mean = float(d.mean())
    return {"n": n, "mean": mean, "se": se, "t": mean / se if se > 0 else float("nan"), "win": int((d > 0).sum())}


CAND = [m for m in ["unrolled_tta", "unrolled", "dncnn_blur"] if m in METHODS]
REFS = [m for m in ["iter1_phys_tta", "iter1_phys", "baseline_unet"] if m in METHODS]

for _ref in REFS:
    print(f"=== 기준: {METHOD_LABEL[_ref]} ===")
    print(f"{'비교':<30}{'subset':<18}{'n':>4}{'ΔPSNR':>9}{'SE':>8}{'t':>8}{'승':>6}")
    print("-" * 84)
    for _c in CAND:
        for _sub in [None, *NOISE_NAMES]:
            st = paired_test(rows, _c, _ref, _sub)
            if st["n"] == 0:
                continue
            name = METHOD_LABEL[_c][:28]
            print(
                f"{name:<30}{(_sub or 'ALL'):<18}{st['n']:>4}{st['mean']:>+9.3f}{st['se']:>8.3f}"
                f"{st['t']:>8.2f}{st['win']:>6}"
            )
        print()

print("|t| > 2 이면 대략 5% 유의수준에서 유의하다. '승' 은 100장(또는 25장) 중 이긴 장수다.")

### 10-2. σ 조건부 입력이 실제로 효과가 있었는가

iter2 의 유일한 목표다. σ 구간별로 분해해서 확인한다.
σ 가 클 때 이득이 커야 설계 의도대로 작동한 것이다.

In [ ]:
if "unrolled" in RESULTS and "iter1_phys" in LOADED:
    _a = "unrolled_tta" if "unrolled_tta" in METHODS else "unrolled"
    _b = "iter1_phys_tta" if "iter1_phys_tta" in METHODS else "iter1_phys"

    _sig = np.array([TEST_NOISE_META.get(r["file"], {}).get("sigma", np.nan) for r in rows])
    _q = np.nanquantile(_sig, [0, 1 / 3, 2 / 3, 1.0])
    print(f"=== sigma 3분위별 ({METHOD_LABEL[_a]} − {METHOD_LABEL[_b]}) ===")
    print(f"{'sigma 구간':<22}{'n':>4}{'iter1':>10}{'iter2':>10}{'Δ':>9}{'SE':>8}{'t':>7}")
    print("-" * 70)
    for _i in range(3):
        _lo, _hi = _q[_i], _q[_i + 1]
        _idx = [j for j, s in enumerate(_sig) if (s >= _lo and (s < _hi or _i == 2))]
        if not _idx:
            continue
        _va = np.mean([rows[j][f"psnr_{_a}"] for j in _idx])
        _vb = np.mean([rows[j][f"psnr_{_b}"] for j in _idx])
        _d = np.array([rows[j][f"psnr_{_a}"] - rows[j][f"psnr_{_b}"] for j in _idx])
        _se = float(np.std(_d, ddof=1) / math.sqrt(len(_d)))
        print(
            f"[{_lo:.4f}, {_hi:.4f}]{'':<4}{len(_idx):>4}{_vb:>10.3f}{_va:>10.3f}"
            f"{_d.mean():>+9.3f}{_se:>8.3f}{_d.mean() / _se if _se > 0 else float('nan'):>7.2f}"
        )
    print()
    print("→ 고σ 구간의 Δ 가 저σ 구간보다 크면 sigma 조건부가 의도대로 작동한 것이다.")

    fig, axes = plt.subplots(1, 2, figsize=(12.5, 4.6))
    for _nt in NOISE_NAMES:
        _x = [TEST_NOISE_META.get(r["file"], {}).get("sigma", np.nan) for r in rows if r["noise_type"] == _nt]
        _y = [r[f"psnr_{_a}"] - r[f"psnr_{_b}"] for r in rows if r["noise_type"] == _nt]
        axes[0].scatter(_x, _y, s=26, alpha=0.75, label=_nt)
    axes[0].axhline(0, color="k", lw=1, ls="--")
    axes[0].set_xlabel("sigma")
    axes[0].set_ylabel("PSNR 개선 [dB]")
    axes[0].set_title("iter2 − iter1 (장별)")
    axes[0].grid(alpha=0.3)
    axes[0].legend(fontsize=8)

    _pos = np.arange(len(NOISE_NAMES))
    axes[1].boxplot(
        [[r[f"psnr_{_a}"] - r[f"psnr_{_b}"] for r in rows if r["noise_type"] == t] for t in NOISE_NAMES],
        positions=_pos,
        widths=0.6,
    )
    axes[1].axhline(0, color="k", lw=1, ls="--")
    axes[1].set_xticks(_pos)
    axes[1].set_xticklabels([t[:9] for t in NOISE_NAMES], fontsize=8)
    axes[1].set_ylabel("PSNR 개선 [dB]")
    axes[1].set_title("noise type 별 개선 분포")
    axes[1].grid(alpha=0.3)
    fig.tight_layout()
    plt.show()
else:
    print("iter2 학습 결과 또는 iter1 체크포인트가 없어 비교를 건너뛴다")

### 10-3. σ ablation (day3 방식을 정확히 재현)

`colab_day3.ipynb` 의 ablation 에서 **"σ 뒤섞음" 이 정상값과 소수점 4자리까지 동일**했다
(29.25 / 0.8777). per-image σ 를 실제로 쓴다면 뒤섞으면 떨어져야 하므로,
그 경로가 모델에 도달하지 않았을 가능성이 높다.

여기서는 σ̂ 를 **모델 내부 입력 텐서에서 직접 치환**해 정확히 측정한다.
`UnrolledHQS._side` 를 임시로 교체하는 방식이므로 우회 경로가 없다.

In [ ]:
if "unrolled" in RESULTS:
    _model = RESULTS["unrolled"]["model"]
    _orig_side = _model._side

    def make_side(mode: str, shuffled: list[float] | None = None):
        def _side(measure: Tensor):
            m, sig_map, neg, imp = _orig_side(measure)
            if mode == "zero":
                sig_map = torch.zeros_like(sig_map)
            elif mode == "double":
                sig_map = sig_map * 2.0
            elif mode == "shuffle" and shuffled is not None:
                b = m.shape[0]
                vals = torch.tensor(shuffled[:b], device=m.device, dtype=m.dtype).view(b, 1, 1, 1)
                sig_map = vals.expand_as(m) * _model.sigma_gain
            return m, sig_map, neg, imp

        return _side

    # 뒤섞기용: test 전체의 sigma_hat 을 미리 모아 한 칸 밀어서 쓴다
    _sig_pool: list[float] = []
    with torch.no_grad():
        for _d in test_loader:
            _mm = _d[DataKey.Measure].to(config.device)
            _sig_pool += estimate_sigma_global(_mm).flatten().tolist()
    _shuf = _sig_pool[1:] + _sig_pool[:1]

    print(f"{'σ 를 어떻게 주는가':<28}{'PSNR':>10}{'SSIM':>10}{'Δ PSNR':>10}")
    print("-" * 60)
    _abl: dict[str, tuple[float, float]] = {}
    for _mode, _label in [
        ("normal", "추정 σ (정상)"),
        ("zero", "σ = 0"),
        ("double", "σ × 2"),
        ("shuffle", "σ 뒤섞음"),
    ]:
        _off = 0
        _model._side = _orig_side if _mode == "normal" else make_side(_mode, _shuf)
        _ps, _ss = [], []
        with torch.no_grad():
            for _d in test_loader:
                _lb = _d[DataKey.Label].to(config.device)
                _mm = _d[DataKey.Measure].to(config.device)
                if _mode == "shuffle":
                    _b = _mm.shape[0]
                    _model._side = make_side("shuffle", _shuf[_off : _off + _b])
                    _off += _b
                _pr = finalize(_model(_mm))
                _ps += calculate_psnr(_pr, _lb).flatten().tolist()
                _ss += calculate_ssim(_pr, _lb).flatten().tolist()
        _abl[_mode] = (float(np.mean(_ps)), float(np.mean(_ss)))
        _d0 = _abl[_mode][0] - _abl["normal"][0]
        print(f"{_label:<28}{_abl[_mode][0]:>10.2f}{_abl[_mode][1]:>10.4f}{_d0:>+10.2f}")

    _model._side = _orig_side
    print()
    print("→ '뒤섞음' 이 정상값과 다르면 per-image σ 가 실제로 쓰이는 것이다.")
    print("→ 동일하다면 모델이 σ 를 per-image 로 활용하지 않는다는 뜻이다 (day3 와 같은 상황).")
else:
    print("unrolled 학습 결과가 없어 건너뛴다")

In [ ]:
# 반복 횟수에 따른 성능 (학습된 가중치 그대로, 반복만 줄인다)
if "unrolled" in RESULTS:
    _model = RESULTS["unrolled"]["model"]
    _T0 = _model.iters
    print(f"{'반복 수':<12}{'PSNR':>10}{'SSIM':>10}")
    print("-" * 34)
    for _T in range(1, _T0 + 1):
        _model.iters = _T
        _ps, _ss = [], []
        with torch.no_grad():
            for _d in test_loader:
                _lb = _d[DataKey.Label].to(config.device)
                _mm = _d[DataKey.Measure].to(config.device)
                _pr = finalize(_model(_mm))
                _ps += calculate_psnr(_pr, _lb).flatten().tolist()
                _ss += calculate_ssim(_pr, _lb).flatten().tolist()
        print(f"{_T:<12}{np.mean(_ps):>10.3f}{np.mean(_ss):>10.4f}")
    _model.iters = _T0
    print()
    print("→ 반복이 늘며 단조 증가하면 unrolling 이 의도대로 작동한 것이다.")
    print("→ 중간에 포화되면 unroll_iters 를 줄여 시간을 절약할 수 있다.")

## 11. 시각화

In [ ]:
BEST_METHOD = max(METHODS, key=lambda k: float(np.mean([r[f"psnr_{k}"] for r in rows])))
print(f"best method: {METHOD_LABEL[BEST_METHOD]} ({BEST_METHOD})")

SHOW = [m for m in ["measure", "conv_median", "baseline_unet", "iter1_phys_tta", "unrolled_tta"] if m in METHODS]
SAMPLE_IDX: int = 0

_label, _measure, _blur, _, _nid, _name = test_ds[SAMPLE_IDX]
_lab_t = _label[None].to(config.device)
_mea_t = _measure[None].to(config.device)
with torch.no_grad():
    _preds = all_predict(_mea_t)

lab = _lab_t.cpu().numpy().squeeze()
n_col = 1 + len(SHOW)
fig, axes = plt.subplots(2, n_col, figsize=(3.0 * n_col, 7.0))
axes[0, 0].imshow(lab, cmap="gray", vmin=0, vmax=1)
axes[0, 0].set_title("Label", fontsize=10)
axes[1, 0].axis("off")
for c, key in enumerate(SHOW):
    img = _preds[key].cpu().numpy().squeeze()
    p = calculate_psnr(_preds[key], _lab_t).item()
    s = calculate_ssim(_preds[key], _lab_t).item()
    err = np.abs(img - lab)
    axes[0, c + 1].imshow(img, cmap="gray", vmin=0, vmax=1)
    axes[0, c + 1].set_title(f"{METHOD_LABEL[key]}\n{p:.2f} dB / {s:.4f}", fontsize=8)
    im = axes[1, c + 1].imshow(err, cmap="magma", vmin=0, vmax=0.3)
    axes[1, c + 1].set_title(f"|error| max {err.max():.3f}", fontsize=8)
for ax in axes.flatten():
    ax.axis("off")
fig.colorbar(im, ax=axes[1, -1], fraction=0.046, pad=0.02)
_nt = TEST_NOISE_META.get(_name, {})
fig.suptitle(f"Restoration | {_name} | {_nt.get('noise_type', '?')} sigma={_nt.get('sigma', float('nan')):.4f}", fontsize=12)
fig.tight_layout(rect=(0, 0, 1, 0.93))
plt.show()

In [ ]:
# 가장 어려운 샘플(최저 PSNR)로 zoom-in
_worst = min(range(len(rows)), key=lambda i: rows[i][f"psnr_{BEST_METHOD}"])
_wname = rows[_worst]["file"]
_widx = test_ds.measure.names.index(_wname)
_lb, _me, _, _, _, _ = test_ds[_widx]
_lb_t, _me_t = _lb[None].to(config.device), _me[None].to(config.device)
with torch.no_grad():
    _wp = all_predict(_me_t)
_wlab = _lb_t.cpu().numpy().squeeze()

ZOOM = (96, 96, 64, 64)
r0, c0, hh, ww = ZOOM
fig, axes = plt.subplots(2, 1 + len(SHOW), figsize=(3.0 * (1 + len(SHOW)), 6.4))
axes[0, 0].imshow(_wlab, cmap="gray", vmin=0, vmax=1)
axes[0, 0].add_patch(plt.Rectangle((c0, r0), ww, hh, ec="lime", fc="none", lw=1.6))
axes[0, 0].set_title("Label", fontsize=9)
axes[1, 0].imshow(_wlab[r0 : r0 + hh, c0 : c0 + ww], cmap="gray", vmin=0, vmax=1)
axes[1, 0].set_title("Label (zoom)", fontsize=9)
for c, key in enumerate(SHOW):
    img = _wp[key].cpu().numpy().squeeze()
    axes[0, c + 1].imshow(img, cmap="gray", vmin=0, vmax=1)
    axes[0, c + 1].add_patch(plt.Rectangle((c0, r0), ww, hh, ec="lime", fc="none", lw=1.6))
    axes[0, c + 1].set_title(METHOD_LABEL[key], fontsize=8)
    axes[1, c + 1].imshow(img[r0 : r0 + hh, c0 : c0 + ww], cmap="gray", vmin=0, vmax=1)
    axes[1, c + 1].set_title(f"{calculate_psnr(_wp[key], _lb_t).item():.2f} dB", fontsize=8)
for ax in axes.flatten():
    ax.axis("off")
_m = TEST_NOISE_META.get(_wname, {})
fig.suptitle(f"최악 샘플 zoom-in | {_wname} | {_m.get('noise_type','?')} sigma={_m.get('sigma',float('nan')):.4f}", fontsize=12)
fig.tight_layout(rect=(0, 0, 1, 0.93))
plt.show()

### 11-1. Error map (상대 오차율 %)

`error_map = (I_restored − I_GT) / I_GT × 100`

`I_GT < 0.05` 인 픽셀은 상대오차가 발산하므로 마스킹한다.

In [ ]:
ERR_DIR = WORK_ROOT / "error_maps"
ERR_DIR.mkdir(parents=True, exist_ok=True)

_pick: dict[str, int] = {}
for _i in range(len(test_ds)):
    _nm = test_ds.measure.names[_i]
    _t = TEST_NOISE_META.get(_nm, {}).get("noise_type")
    if _t in NOISE_NAMES and _t not in _pick:
        _pick[_t] = _i
    if len(_pick) == len(NOISE_NAMES):
        break

fig, axes = plt.subplots(3, len(_pick), figsize=(4.0 * len(_pick), 11.5))
if len(_pick) == 1:
    axes = axes[:, None]
for c, (_t, _i) in enumerate(sorted(_pick.items())):
    _lb, _me, _, _, _, _nm = test_ds[_i]
    _lb_t, _me_t = _lb[None].to(config.device), _me[None].to(config.device)
    with torch.no_grad():
        _pr = all_predict(_me_t)[BEST_METHOD]
    gt = _lb_t.cpu().numpy().squeeze()
    pr = _pr.cpu().numpy().squeeze()
    rel = np.ma.masked_where(gt < 0.05, (pr - gt) / (gt + 1e-6) * 100.0)

    axes[0, c].imshow(gt, cmap="gray", vmin=0, vmax=1)
    axes[0, c].set_title(f"GT | {_t}\nsigma={TEST_NOISE_META.get(_nm, {}).get('sigma', float('nan')):.4f}", fontsize=9)
    axes[1, c].imshow(pr, cmap="gray", vmin=0, vmax=1)
    axes[1, c].set_title(f"restored  {calculate_psnr(_pr, _lb_t).item():.2f} dB / {calculate_ssim(_pr, _lb_t).item():.4f}", fontsize=9)
    im = axes[2, c].imshow(rel, cmap="magma", vmin=-20, vmax=20)
    axes[2, c].set_title(f"relative error [%]  median|e|={np.ma.median(np.abs(rel)):.2f}%", fontsize=9)
    plt.colorbar(im, ax=axes[2, c], fraction=0.046, pad=0.02)
    for r in range(3):
        axes[r, c].axis("off")
fig.suptitle(f"Error map: (restored − GT)/GT × 100  |  {METHOD_LABEL[BEST_METHOD]}", fontsize=13)
fig.tight_layout(rect=(0, 0, 1, 0.95))
fig.savefig(ERR_DIR / f"error_maps_iter3_{BEST_METHOD}.png", dpi=130, bbox_inches="tight")
plt.show()
print(f"저장: {ERR_DIR / f'error_maps_iter3_{BEST_METHOD}.png'}")

## 12. 취약점 분석 & 요약

In [ ]:
_key = BEST_METHOD
print(f"=== {METHOD_LABEL[_key]} ===")
print()
print("[noise type 별]")
for _t in sorted(NOISE_NAMES, key=lambda t: np.mean([r[f"psnr_{_key}"] for r in rows if r["noise_type"] == t])):
    _v = [r[f"psnr_{_key}"] for r in rows if r["noise_type"] == _t]
    _s = [r[f"ssim_{_key}"] for r in rows if r["noise_type"] == _t]
    _in = [r["psnr_measure"] for r in rows if r["noise_type"] == _t]
    print(f"  {_t:<18} PSNR {np.mean(_v):>7.3f} (입력 {np.mean(_in):>6.3f}, 개선 {np.mean(_v) - np.mean(_in):>+6.3f})  SSIM {np.mean(_s):.4f}")

print()
print("[하위 5장]")
for r in sorted(rows, key=lambda r: r[f"psnr_{_key}"])[:5]:
    _m = TEST_NOISE_META.get(r["file"], {})
    print(f"  {r['file'][:32]:<34}{r['noise_type']:<18}sigma={_m.get('sigma', float('nan')):.4f}  PSNR {r[f'psnr_{_key}']:.3f}")
print()
print("[상위 5장]")
for r in sorted(rows, key=lambda r: -r[f"psnr_{_key}"])[:5]:
    _m = TEST_NOISE_META.get(r["file"], {})
    print(f"  {r['file'][:32]:<34}{r['noise_type']:<18}sigma={_m.get('sigma', float('nan')):.4f}  PSNR {r[f'psnr_{_key}']:.3f}")

In [ ]:
if RESULTS:
    fig, axes = plt.subplots(1, 3, figsize=(16, 4.4))
    for _k, _r in RESULTS.items():
        _h = [x for x in _r["history"] if "val_psnr" in x]
        if not _h:
            continue
        axes[0].plot([x["epoch"] for x in _h], [x["val_psnr"] for x in _h], marker="o", ms=2.5, label=_k)
        axes[1].plot([x["epoch"] for x in _r["history"]], [x["train_loss"] for x in _r["history"]], label=_k)
        axes[2].plot([x["epoch"] for x in _r["history"]], [x["lr"] for x in _r["history"]], label=_k)
    # iter1 C 의 최종 val 을 수평선으로
    if "iter1_phys" in LOADED and LOADED["iter1_phys"].get("best"):
        _b1 = LOADED["iter1_phys"]["best"].get("psnr")
        if _b1:
            axes[0].axhline(_b1, color="r", ls="--", lw=1, label=f"iter1 C best ({_b1:.2f})")
    axes[0].set_title("stratified val PSNR")
    axes[0].set_ylabel("PSNR [dB]")
    axes[1].set_title("train loss")
    axes[1].set_yscale("log")
    axes[2].set_title("learning rate")
    axes[2].set_yscale("log")
    for a in axes:
        a.set_xlabel("epoch")
        a.grid(alpha=0.3)
        a.legend(fontsize=8)
    fig.tight_layout()
    plt.show()

print()
print(f"{'run':<14}{'params':>12}{'best ep':>9}{'val PSNR':>11}{'test PSNR':>11}{'test SSIM':>11}")
print("-" * 68)
for _k, _r in RESULTS.items():
    _tp = np.mean([r[f"psnr_{_k}"] for r in rows]) if f"psnr_{_k}" in rows[0] else float("nan")
    _ts = np.mean([r[f"ssim_{_k}"] for r in rows]) if f"ssim_{_k}" in rows[0] else float("nan")
    print(f"{_k:<14}{_r['params']:>12,}{_r['best']['epoch']:>9}{_r['best']['psnr']:>11.3f}{_tp:>11.3f}{_ts:>11.4f}")
for _k, _v in LOADED.items():
    _tp = np.mean([r[f"psnr_{_k}"] for r in rows]) if f"psnr_{_k}" in rows[0] else float("nan")
    _ts = np.mean([r[f"ssim_{_k}"] for r in rows]) if f"ssim_{_k}" in rows[0] else float("nan")
    _be = _v.get("best", {}) or {}
    print(f"{_k:<14}{_v['params']:>12,}{_be.get('epoch', -1):>9}{_be.get('psnr', float('nan')):>11.3f}{_tp:>11.3f}{_ts:>11.4f}")

In [ ]:
summary = {
    "iter": 3,
    "timestamp": datetime.now().isoformat(timespec="seconds"),
    "change": ["unrolled HQS (4 iter, closed-form DC)", "rician frac_neg + impulse map", "SSIM 0.5 + Sobel 0.1", "deep supervision"],
    "config": {k: str(v) for k, v in asdict(config).items()},
    "best_wiener_K": BEST_K,
    "dncnn_deconv_K": RESULTS.get("dncnn_blur", {}).get("deconv_K"),
    "rule_compliance": {
        "variant_mode": config.variant_mode,
        "noise_realizations_per_clean": config.variant_pool if config.variant_mode == "pool" else "unbounded",
        "augment_flip": config.augment_flip,
        "distinct_corrupted_per_clean": (
            config.variant_pool * (4 if config.augment_flip else 1) if config.variant_mode == "pool" else "unbounded"
        ),
        "limit": RULE_MAX_VARIANTS,
        "strictly_compliant": config.variant_mode == "pool"
        and not config.augment_flip
        and config.variant_pool <= RULE_MAX_VARIANTS,
    },
    "best_method": BEST_METHOD,
    "test": {
        k: {
            "psnr": round(float(np.mean([r[f"psnr_{k}"] for r in rows])), 3),
            "ssim": round(float(np.mean([r[f"ssim_{k}"] for r in rows])), 4),
            "per_noise_psnr": {
                t: round(float(np.mean([r[f"psnr_{k}"] for r in rows if r["noise_type"] == t])), 3)
                for t in NOISE_NAMES
                if any(r["noise_type"] == t for r in rows)
            },
        }
        for k in METHODS
    },
    "paired_vs_iter1": (
        {
            m: {
                (s or "ALL"): {kk: (round(vv, 4) if isinstance(vv, float) else vv) for kk, vv in paired_test(rows, m, "iter1_phys_tta", s).items()}
                for s in [None, *NOISE_NAMES]
            }
            for m in CAND
        }
        if "iter1_phys_tta" in METHODS
        else {}
    ),
    "runs": {
        k: {"run_dir": str(v["run_dir"]), "best": v["best"], "params": v["params"], "spec": v["spec"]}
        for k, v in RESULTS.items()
    },
}
_out = WORK_ROOT / "summary_iter3.json"
_out.write_text(json.dumps(summary, indent=2, ensure_ascii=False), encoding="utf-8")
print(f"저장: {_out}")
print()
print(json.dumps(summary["test"], indent=2, ensure_ascii=False))

## 13. 다음 단계

이 노트북의 수치를 보고 판단한다.

| 관측 | 해석 | 조치 |
|---|---|---|
| `unrolled` > `iter1_phys` 이고 고σ 구간 이득이 큼 | σ 조건부가 의도대로 작동 | 채택. `sigma_gain` / `local_block` 튜닝 |
| 전체는 개선인데 고σ 구간 이득이 없음 | σ̂ 가 정보를 주지 못함 (MAD 과대추정) | σ̂ 추정을 wavelet 기반으로 교체 |
| `unrolled` ≈ `iter1_phys` (|t| < 2) | σ̂ 채널 무효 또는 `dc_weight` 제거가 상쇄 | `dc_weight` 만 되돌려 분리 검증 |
| `unrolled` < `iter1_phys` | σ̂ 채널이 방해 | `sigma_gain` 1.0 으로 낮추거나 global 만 사용 |
| `dncnn_blur` 가 `iter1_phys` 에 근접 | 역할 분리 접근도 유효 | **앙상블** (두 출력 평균) 시도 |

### 남은 후보

| # | 항목 | 근거 | 비용 |
|---|---|---|---|
| 1 | **앙상블** (`unrolled` + `dncnn_blur` 평균) | 서로 다른 구조의 오차는 상관이 낮다 | 10분 |
| 2 | **Unrolled / physics DC step** | 제공 `model_config` 의 `unroll_iters=5`, `phys_dc_steps`, `phys_dc_eta=0.1` 흔적. 역문제에서 표준 SOTA | 4h |
| 3 | Label-free (Noise2Noise) | 가산점. 12개 풀의 같은 noise type 두 실현을 pair 로 | 3h |
| 4 | Rician 부호 복원 | 부호 소실 픽셀이 4.8%. 소규모지만 남은 rician 손실(2.58 dB)의 일부 | 3h |

### 하지 말아야 할 것

- **Rician 2차 모멘트 debias**: 헤더에서 실측으로 역효과 확인
- **epoch 증량**: iter1 에서 ep 49~60 표준편차 0.019 dB 로 수렴 확인
- **ALL 평균만 보고 판단**: noise type 과 σ 구간으로 분해해야 원인이 보인다